In [ ]:
# Cấu hình số luồng CPU trước khi import NumPy/Pandas/TensorFlow.
import os

logical_cpus = os.cpu_count() or 1
compute_threads = max(1, logical_cpus // 2)
interop_threads = min(8, max(1, logical_cpus // 4))

os.environ['OPENBLAS_NUM_THREADS'] = str(compute_threads)
os.environ['MKL_NUM_THREADS'] = str(compute_threads)
os.environ['OMP_NUM_THREADS'] = str(compute_threads)
os.environ['NUMEXPR_NUM_THREADS'] = str(compute_threads)
os.environ['TF_NUM_INTRAOP_THREADS'] = str(compute_threads)
os.environ['TF_NUM_INTEROP_THREADS'] = str(interop_threads)

# os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print(f'CPU logic: {logical_cpus}')
print(f'BLAS/TF intra-op: {compute_threads} luồng')
print(f'TF inter-op: {interop_threads} luồng')

In [ ]:
%%time

# =========================================================
# BƯỚC 5: ĐỌC DỮ LIỆU VÀ MÃ HÓA NHÃN LABEL
# =========================================================

import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder

step4_input_file = os.path.abspath(os.path.join(
    "output_after_preprocess",
    "dataset_after_labeling.csv"
))

if not os.path.isfile(step4_input_file):
    raise FileNotFoundError(
        f"Không tìm thấy file: {step4_input_file}"
    )

print(f"Đang đọc dữ liệu từ: {step4_input_file}")

dataset = pd.read_csv(
    step4_input_file,
    low_memory=False
)

print(f"Đã đọc {dataset.shape[0]:,} dòng và {dataset.shape[1]} cột.")

if "Label" not in dataset.columns:
    raise KeyError("Dữ liệu không có cột 'Label'.")

# ---------------------------------------------------------
# Timestamp: CHỈ dùng để chia dữ liệu, KHÔNG dùng làm đặc trưng
# ---------------------------------------------------------
# Trong CSE-CIC-IDS2018, các đợt tấn công diễn ra trong những khung giờ
# cố định, và sau khi lọc ở Bước 4 thì hai ngày 15/02 và 16/02 chỉ còn
# lại lưu lượng Benign. Nếu đưa Timestamp vào X, mô hình chỉ cần học
# "lịch tấn công" là đạt gần 100% mà không hề học đặc trưng mạng nào -
# kết quả đó vô dụng khi triển khai thật.
# Cột này bị loại khỏi X ở Bước 6, chỉ giữ để chia train/test ở Bước 7.

if "Timestamp" not in dataset.columns:
    raise KeyError(
        "Dữ liệu không có cột 'Timestamp'. "
        "Cần cột này để chia train/test theo thời gian."
    )

if not pd.api.types.is_numeric_dtype(dataset["Timestamp"]):
    timestamp_parsed = pd.to_datetime(
        dataset["Timestamp"],
        errors="coerce",
        dayfirst=True
    )
    invalid_timestamp_count = int(timestamp_parsed.isna().sum())

    if invalid_timestamp_count > 0:
        print(f"Loại {invalid_timestamp_count:,} dòng vì Timestamp không hợp lệ.")
        valid_timestamp_mask = timestamp_parsed.notna()
        dataset = dataset.loc[valid_timestamp_mask].copy()
        timestamp_parsed = timestamp_parsed.loc[valid_timestamp_mask]

    dataset["Timestamp"] = (
        timestamp_parsed.astype("int64") // 10**9
    ).astype("int64")
    dataset.reset_index(drop=True, inplace=True)
    print("Đã mã hóa Timestamp thành Unix time.")
else:
    print("Timestamp đã là dạng số.")

# Cột ngày dùng để chia dữ liệu theo thời gian ở Bước 7.
dataset["Day"] = pd.to_datetime(
    dataset["Timestamp"],
    unit="s"
).dt.strftime("%Y-%m-%d")

# ---------------------------------------------------------
# Tên biến thể tấn công gốc (chỉ dùng để báo cáo ở Bước 12)
# ---------------------------------------------------------
if "Attack Type" not in dataset.columns:
    print(
        "CẢNH BÁO: không tìm thấy cột 'Attack Type'. "
        "Hãy chạy lại Bước 4 của preprocessing_data.ipynb để có báo cáo "
        "recall theo từng biến thể tấn công. Tạm dùng Label thay thế."
    )
    dataset["Attack Type"] = dataset["Label"]

dataset["Attack Type"] = dataset["Attack Type"].astype(str).str.strip()

# ---------------------------------------------------------
# Mã hóa nhãn nhị phân
# ---------------------------------------------------------
dataset["Label"] = dataset["Label"].astype(str).str.strip()

label_encoder = LabelEncoder()
dataset["Label"] = label_encoder.fit_transform(dataset["Label"])

label_mapping = dict(zip(
    label_encoder.classes_,
    label_encoder.transform(label_encoder.classes_)
))

inverse_label_mapping = {
    int(code): name
    for name, code in label_mapping.items()
}

print("Bảng mã hóa nhãn:")
for label, number in label_mapping.items():
    print(f"{label} -> {number}")

print("\nPhân bố nhãn theo từng ngày:")
display(pd.crosstab(dataset["Day"], dataset["Label"]))

print("\nPhân bố biến thể tấn công theo từng ngày:")
display(pd.crosstab(dataset["Day"], dataset["Attack Type"]))

print("\nKích thước dữ liệu sau Bước 5:", dataset.shape)
display(dataset.head())

In [ ]:
%%time

# =========================================================
# BƯỚC 6: CHỌN ĐẶC TRƯNG X VÀ NHÃN y
# =========================================================

import gc
import hashlib
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Cấu hình chống rò rỉ dữ liệu
# ---------------------------------------------------------

# Các cột KHÔNG bao giờ được dùng làm đặc trưng.
#   Label       : nhãn cần dự đoán.
#   Timestamp   : rò rỉ lịch tấn công (xem giải thích ở Bước 5).
#   Day         : dẫn xuất từ Timestamp, chỉ dùng để chia dữ liệu.
#   Attack Type : tên tấn công gốc, chỉ dùng để báo cáo ở Bước 12.
EXCLUDED_COLUMNS = [
    "Label", "Timestamp", "Day", "Attack Type",
    "Init Fwd Win Byts", "Init Bwd Win Byts",
]

# Dst Port cũng dễ gây rò rỉ vì trong bộ dữ liệu này các đợt tấn công
# đều nhắm vào một máy nạn nhân với cổng cố định (80/443).
# Đặt True để chạy thí nghiệm ablation loại bỏ cột này.
DROP_DST_PORT = True

# ---------------------------------------------------------
# Danh sách đặc trưng ứng viên
# ---------------------------------------------------------

feature_candidates = [
    column
    for column in dataset.columns
    if column not in EXCLUDED_COLUMNS
]

if DROP_DST_PORT and "Dst Port" in feature_candidates:
    feature_candidates.remove("Dst Port")
    print("Đã loại 'Dst Port' theo cấu hình DROP_DST_PORT.")

non_numeric_columns = (
    dataset[feature_candidates]
    .select_dtypes(exclude=["number", "bool"])
    .columns
    .tolist()
)

if non_numeric_columns:
    raise TypeError(
        f"Còn cột không phải dạng số trong đặc trưng: {non_numeric_columns}"
    )

# ---------------------------------------------------------
# Loại cột hằng số (không mang thông tin phân biệt)
# ---------------------------------------------------------
# Ví dụ trong bộ này: Bwd PSH Flags, Fwd/Bwd URG Flags, CWE Flag Count,
# Fwd/Bwd Byts/b Avg, Fwd/Bwd Pkts/b Avg, Fwd/Bwd Blk Rate Avg
# đều bằng 0 ở mọi dòng.

column_min = dataset[feature_candidates].min()
column_max = dataset[feature_candidates].max()
constant_columns = column_min.index[column_min == column_max].tolist()

if constant_columns:
    print(f"\nLoại {len(constant_columns)} cột hằng số:")
    for column in constant_columns:
        print(f"  - {column}")
    feature_candidates = [
        column
        for column in feature_candidates
        if column not in constant_columns
    ]
else:
    print("\nKhông có cột hằng số.")

# ---------------------------------------------------------
# Loại cột trùng lặp hoàn toàn về giá trị
# ---------------------------------------------------------
# Ví dụ: Fwd Seg Size Avg trùng Fwd Pkt Len Mean,
#        Subflow Fwd Pkts trùng Tot Fwd Pkts, ...
# So sánh bằng chữ ký băm để không phải so từng cặp cột.

column_signatures = {}
duplicate_columns = []

for column in feature_candidates:
    values = np.ascontiguousarray(
        dataset[column].to_numpy(dtype=np.float64)
    )
    signature = hashlib.md5(values.tobytes()).hexdigest()

    if signature in column_signatures:
        duplicate_columns.append((column, column_signatures[signature]))
    else:
        column_signatures[signature] = column

if duplicate_columns:
    print(f"\nLoại {len(duplicate_columns)} cột trùng lặp:")
    for column, kept_column in duplicate_columns:
        print(f"  - {column} (trùng với {kept_column})")
    dropped = {column for column, _ in duplicate_columns}
    feature_candidates = [
        column
        for column in feature_candidates
        if column not in dropped
    ]
else:
    print("\nKhông có cột trùng lặp.")

feature_names = feature_candidates

# ---------------------------------------------------------
# Tạo mảng dữ liệu cuối cùng
# ---------------------------------------------------------

# float32 giúp giảm khoảng một nửa bộ nhớ so với float64.
X_all = dataset[feature_names].to_numpy(dtype=np.float32)
y_all = dataset["Label"].to_numpy(dtype=np.int8)

# Hai mảng dưới đây KHÔNG phải đặc trưng, chỉ phục vụ việc chia dữ liệu
# theo thời gian ở Bước 7.
day_all = dataset["Day"].to_numpy()
timestamp_all = dataset["Timestamp"].to_numpy(dtype=np.int64)

# Mã hóa biến thể tấn công thành số để tiết kiệm bộ nhớ.
attack_type_categorical = dataset["Attack Type"].astype("category")
attack_type_names = attack_type_categorical.cat.categories.tolist()
attack_type_codes_all = attack_type_categorical.cat.codes.to_numpy(dtype=np.int8)

# Giải phóng DataFrame gốc để tiết kiệm RAM cho các bước sau.
del dataset
gc.collect()

print("\n=========================================================")
print(f"Số đặc trưng dùng để train: {len(feature_names)}")
print("Kích thước X_all:", X_all.shape)
print("Kích thước y_all:", y_all.shape)
print("Kiểu dữ liệu X_all:", X_all.dtype)
print("\nCác cột đã loại khỏi X:")
print(f"  - Rò rỉ / phụ trợ: {EXCLUDED_COLUMNS}")
print(f"  - Hằng số: {constant_columns or 'Không có'}")
print(
    "  - Trùng lặp: "
    f"{[column for column, _ in duplicate_columns] or 'Không có'}"
)
print("\nDanh sách đặc trưng cuối cùng:")
display(pd.DataFrame({
    "STT": np.arange(1, len(feature_names) + 1),
    "Feature": feature_names
}))

In [ ]:
%%time

# =========================================================
# BƯỚC 7: CHIA DỮ LIỆU THÀNH TẬP TRAIN VÀ TEST
# =========================================================

import gc
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split

# ---------------------------------------------------------
# Ba chế độ chia dữ liệu - chạy cả ba để so sánh
# ---------------------------------------------------------
#
# 'random'
#     Chia ngẫu nhiên có phân tầng. Dùng để đối chiếu và để trả lời câu
#     hỏi "mô hình có học được đặc trưng mạng thật không, khi đã bỏ
#     Timestamp?". Con số sẽ lạc quan vì các flow rất giống nhau trong
#     cùng một đợt tấn công bị chia vào cả hai tập.
#
# 'temporal_within_day'
#     Trong MỖI ngày và MỖI loại tấn công, lấy phần sớm nhất theo thời
#     gian làm train và phần muộn nhất làm test. Mọi loại tấn công đều
#     có mặt ở cả hai tập, nhưng test luôn nằm sau train về thời gian.
#     Đây là kịch bản "phát hiện tấn công đã biết trên lưu lượng mới" -
#     con số nên dùng làm kết quả chính trong báo cáo.
#
# 'temporal_by_day'
#     Train trên ngày này, test trên ngày khác. Vì mỗi ngày là một loại
#     tấn công khác nhau, đây là kịch bản khó nhất: "phát hiện loại tấn
#     công CHƯA từng thấy" (zero-day). Kết quả thấp ở đây KHÔNG có nghĩa
#     là pipeline sai, mà là mô hình không tổng quát hóa sang họ tấn công
#     mới - đó cũng là một kết luận đáng báo cáo.
SPLIT_MODE = "temporal_within_day"

# Cấu hình cho 'temporal_by_day'
# Thứ Năm 15/02 và Thứ Sáu 16/02 sau khi lọc chỉ còn lưu lượng Benign.
# Thứ Ba  20/02: DDoS attacks-LOIC-HTTP
# Thứ Tư  21/02: DDOS attack-HOIC và DDOS attack-LOIC-UDP
TRAIN_DAYS = ["2018-02-15", "2018-02-20"]
TEST_DAYS = ["2018-02-16", "2018-02-21"]

# Tỷ lệ tập test cho hai chế độ còn lại
TEST_SIZE = 0.2
RANDOM_STATE = 42

available_days = sorted(pd.unique(day_all))
print("Các ngày có trong dữ liệu:", available_days)

if SPLIT_MODE == "temporal_by_day":
    missing_days = [
        day
        for day in TRAIN_DAYS + TEST_DAYS
        if day not in available_days
    ]

    if missing_days:
        raise ValueError(
            f"Không tìm thấy dữ liệu cho các ngày: {missing_days}. "
            f"Các ngày hiện có: {available_days}"
        )

    unassigned_days = [
        day
        for day in available_days
        if day not in TRAIN_DAYS and day not in TEST_DAYS
    ]

    if unassigned_days:
        print(
            "CẢNH BÁO: các ngày sau không thuộc train lẫn test nên bị bỏ "
            f"qua: {unassigned_days}"
        )

    train_index = np.flatnonzero(np.isin(day_all, TRAIN_DAYS))
    test_index = np.flatnonzero(np.isin(day_all, TEST_DAYS))

    SPLIT_DESCRIPTION = (
        f"chia theo ngày: train {TRAIN_DAYS} / test {TEST_DAYS} "
        "(loại tấn công trong test chưa từng xuất hiện khi train)"
    )

elif SPLIT_MODE == "temporal_within_day":
    # Cắt theo thời gian bên trong từng nhóm (ngày, loại tấn công) để
    # mọi loại tấn công đều có mặt ở cả train lẫn test theo đúng tỷ lệ.
    train_blocks = []
    test_blocks = []

    group_frame = pd.DataFrame({
        "day": day_all,
        "attack": attack_type_codes_all
    })

    for (day_value, attack_code), group in group_frame.groupby(
        ["day", "attack"],
        sort=True
    ):
        group_positions = group.index.to_numpy()
        order = np.argsort(timestamp_all[group_positions], kind="stable")
        ordered_positions = group_positions[order]

        cut_position = int(round(len(ordered_positions) * (1 - TEST_SIZE)))
        cut_position = min(max(cut_position, 1), len(ordered_positions) - 1)

        train_blocks.append(ordered_positions[:cut_position])
        test_blocks.append(ordered_positions[cut_position:])

        print(
            f"  {day_value} | {attack_type_names[attack_code]:<24} "
            f"tổng {len(ordered_positions):>9,} -> "
            f"train {cut_position:>9,} / test "
            f"{len(ordered_positions) - cut_position:>9,}"
        )

    train_index = np.sort(np.concatenate(train_blocks))
    test_index = np.sort(np.concatenate(test_blocks))

    del group_frame
    gc.collect()

    SPLIT_DESCRIPTION = (
        f"chia theo thời gian trong ngày: {(1 - TEST_SIZE) * 100:.0f}% sớm "
        f"nhất để train / {TEST_SIZE * 100:.0f}% muộn nhất để test"
    )

elif SPLIT_MODE == "random":
    train_index, test_index = train_test_split(
        np.arange(len(y_all)),
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True,
        stratify=y_all
    )

    SPLIT_DESCRIPTION = (
        f"chia ngẫu nhiên có phân tầng "
        f"{(1 - TEST_SIZE) * 100:.0f}% / {TEST_SIZE * 100:.0f}%"
    )

else:
    raise ValueError(
        "SPLIT_MODE phải là 'random', 'temporal_within_day' hoặc "
        f"'temporal_by_day', nhận được: {SPLIT_MODE}"
    )

print(f"\nChế độ chia dữ liệu: {SPLIT_MODE}")
print(f"  {SPLIT_DESCRIPTION}")

# ---------------------------------------------------------
# Tách dữ liệu theo chỉ số
# ---------------------------------------------------------
# Tập test GIỮ NGUYÊN phân bố nhãn tự nhiên, không cân bằng lại,
# để các chỉ số precision/FPR phản ánh đúng khi triển khai.

X_train_full = X_all[train_index]
y_train_full = y_all[train_index]

X_test = X_all[test_index]
y_test = y_all[test_index]
attack_type_codes_test = attack_type_codes_all[test_index]
day_test = day_all[test_index]

del X_all
gc.collect()

for subset_name, subset_labels in (
    ("TRAIN", y_train_full),
    ("TEST", y_test)
):
    if len(np.unique(subset_labels)) != 2:
        raise ValueError(
            f"Tập {subset_name} không có đủ 2 lớp: "
            f"{np.unique(subset_labels).tolist()}"
        )

print("\nKích thước X_train_full:", X_train_full.shape)
print("Kích thước X_test      :", X_test.shape)

for subset_name, subset_labels in (
    ("TRAIN", y_train_full),
    ("TEST", y_test)
):
    counts = Counter(subset_labels.tolist())
    total = len(subset_labels)

    print(f"\nPhân bố nhãn tập {subset_name} (tổng {total:,}):")
    for name, code in sorted(label_mapping.items(), key=lambda item: item[1]):
        count = counts.get(int(code), 0)
        print(f"  {code} ({name}): {count:,} ({count / total * 100:.2f}%)")

    imbalance_ratio = (
        counts.get(0, 0) / counts.get(1, 1)
        if counts.get(1, 0) > 0
        else float("inf")
    )
    print(f"  Tỷ lệ Benign/DDoS: {imbalance_ratio:.2f} : 1")

print("\nBiến thể tấn công trong tập TRAIN:")
display(pd.Series(
    [attack_type_names[code] for code in attack_type_codes_all[train_index]]
).value_counts())

print("Biến thể tấn công trong tập TEST:")
display(pd.Series(
    [attack_type_names[code] for code in attack_type_codes_test]
).value_counts())

In [ ]:
%%time

# =========================================================
# BƯỚC 8: CHUẨN HÓA ĐẶC TRƯNG (CHỈ HỌC THAM SỐ TỪ TẬP TRAIN)
# =========================================================

import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# StandardScaler phải được fit CHỈ trên tập train.
# Nếu fit trên toàn bộ dữ liệu trước khi chia, giá trị trung bình và
# độ lệch chuẩn của tập test sẽ rò rỉ vào quá trình huấn luyện.

if len(feature_names) != X_train_full.shape[1]:
    raise ValueError(
        f"Số feature không khớp: feature_names={len(feature_names)}, "
        f"X_train_full={X_train_full.shape[1]}"
    )

scaler = StandardScaler()
scaler.fit(X_train_full)

X_train_full = scaler.transform(X_train_full).astype(np.float32, copy=False)
X_test = scaler.transform(X_test).astype(np.float32, copy=False)

print("Đã chuẩn hóa dữ liệu bằng tham số học từ tập TRAIN.")
print("Kích thước X_train_full:", X_train_full.shape)
print("Kích thước X_test      :", X_test.shape)
print("Kiểu dữ liệu:", X_train_full.dtype)

# ---------------------------------------------------------
# Lưu scaler và danh sách feature để dùng lại khi triển khai
# ---------------------------------------------------------
# Scaler phụ thuộc vào cách chia dữ liệu nên được lưu theo từng
# SPLIT_MODE, cùng chỗ với model tương ứng.

model_output_dir = os.path.abspath(
    os.path.join("output_after_preprocess", "dnn_model", SPLIT_MODE)
)
os.makedirs(model_output_dir, exist_ok=True)

joblib.dump(
    scaler,
    os.path.join(model_output_dir, "standard_scaler.joblib")
)

with open(
    os.path.join(model_output_dir, "feature_names.json"),
    "w",
    encoding="utf-8"
) as feature_file:
    json.dump(feature_names, feature_file, ensure_ascii=False, indent=2)

print(f"\nĐã lưu scaler và danh sách feature tại: {model_output_dir}")

# ---------------------------------------------------------
# Kiểm tra độ lệch phân bố giữa train và test
# ---------------------------------------------------------
# Giá trị z quá lớn ở tập test cho thấy phân bố đặc trưng của phần test
# khác hẳn phần train. Đây là thông tin quan trọng khi chia theo thời gian:
# nếu lệch quá nhiều thì mô hình gần như chắc chắn sẽ tổng quát hóa kém.

max_absolute_z = np.abs(X_test).max(axis=0)
shift_report = pd.DataFrame({
    "Feature": feature_names,
    "Max |z| trên test": max_absolute_z
}).sort_values("Max |z| trên test", ascending=False)

print("\n10 đặc trưng lệch phân bố nhiều nhất giữa train và test:")
display(shift_report.head(10).reset_index(drop=True))

mean_scale_df = pd.DataFrame({
    "STT": np.arange(1, len(feature_names) + 1),
    "Feature": feature_names,
    "Mean (train)": scaler.mean_,
    "Scale (train)": scaler.scale_,
})

print(f"\nBảng Mean và Scale của {len(feature_names)} feature:")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(mean_scale_df)

In [ ]:
%%time

# =========================================================
# BƯỚC 9: TÁCH VALIDATION VÀ CÂN BẰNG LỚP (CHỈ TRÊN TRAIN)
# =========================================================

import gc
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler

VALIDATION_SIZE = 0.1
RANDOM_STATE = 42

# ---------------------------------------------------------
# 1. Tách tập validation ra khỏi train
# ---------------------------------------------------------
# Validation GIỮ NGUYÊN phân bố tự nhiên (không cân bằng lại) vì nó
# được dùng để dừng sớm và để dò ngưỡng quyết định ở Bước 12.
# Nếu cân bằng validation, ngưỡng tìm được sẽ không đúng với tỷ lệ
# Benign/DDoS thật khi triển khai.

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=y_train_full
)

del X_train_full, y_train_full
gc.collect()

# ---------------------------------------------------------
# 2. Cân bằng lớp bằng undersampling - CHỈ trên tập train
# ---------------------------------------------------------
# Trước đây undersampling được chạy trên toàn bộ dữ liệu rồi mới chia,
# khiến tập test cũng bị ép về 50/50 và làm precision đẹp hơn thực tế.
# Giờ chỉ tập train được cân bằng; validation và test giữ nguyên.

counts_before = Counter(y_train.tolist())

print("Phân bố nhãn tập TRAIN trước khi cân bằng:")
for name, code in sorted(label_mapping.items(), key=lambda item: item[1]):
    print(f"  {code} ({name}): {counts_before.get(int(code), 0):,}")

under_sampler = RandomUnderSampler(
    sampling_strategy="auto",
    random_state=RANDOM_STATE
)

X_train, y_train = under_sampler.fit_resample(X_train, y_train)

X_train = np.asarray(X_train, dtype=np.float32)
y_train = np.asarray(y_train, dtype=np.int8)

gc.collect()

counts_after = Counter(y_train.tolist())

print("\nPhân bố nhãn tập TRAIN sau khi cân bằng:")
for name, code in sorted(label_mapping.items(), key=lambda item: item[1]):
    count = counts_after.get(int(code), 0)
    print(f"  {code} ({name}): {count:,} ({count / len(y_train) * 100:.2f}%)")

# ---------------------------------------------------------
# 3. Tổng kết ba tập dữ liệu
# ---------------------------------------------------------

print("\n=========================================================")
for subset_name, subset_labels, note in (
    ("TRAIN", y_train, "đã cân bằng 50/50"),
    ("VALIDATION", y_val, "giữ phân bố tự nhiên"),
    ("TEST", y_test, "giữ phân bố tự nhiên")
):
    counts = Counter(subset_labels.tolist())
    total = len(subset_labels)
    benign = counts.get(0, 0)
    ddos = counts.get(1, 0)

    print(
        f"{subset_name:<11} {total:>10,} mẫu | "
        f"Benign {benign:>10,} | DDoS {ddos:>10,} | {note}"
    )

print("\nKích thước X_train:", X_train.shape)
print("Kích thước X_val  :", X_val.shape)
print("Kích thước X_test :", X_test.shape)

for subset_name, subset_labels in (
    ("y_train", y_train),
    ("y_val", y_val),
    ("y_test", y_test)
):
    if len(np.unique(subset_labels)) != 2:
        raise ValueError(
            f"{subset_name} không có đủ 2 lớp: "
            f"{np.unique(subset_labels).tolist()}"
        )

In [ ]:
%%time

# =========================================================
# BƯỚC 10: CHUẨN BỊ DỮ LIỆU 2 LỚP CHO DNN
# =========================================================

import numpy as np

NUM_CLASSES = 2
NUM_FEATURES = X_train.shape[1]

# Kiểm tra dữ liệu chỉ gồm hai nhãn 0 và 1
for subset_name, subset_labels in (
    ("y_train", y_train),
    ("y_val", y_val),
    ("y_test", y_test)
):
    unique_labels = np.unique(subset_labels)
    if not np.array_equal(unique_labels, [0, 1]):
        raise ValueError(
            f"{subset_name} phải chứa nhãn [0, 1], hiện có: {unique_labels}"
        )

# DNN nhận dữ liệu 2 chiều: (samples, features)
X_train_dnn = np.ascontiguousarray(X_train, dtype=np.float32)
X_val_dnn = np.ascontiguousarray(X_val, dtype=np.float32)
X_test_dnn = np.ascontiguousarray(X_test, dtype=np.float32)

# Binary crossentropy không cần one-hot encoding
y_train_dnn = np.asarray(y_train, dtype=np.float32)
y_val_dnn = np.asarray(y_val, dtype=np.float32)
y_test_dnn = np.asarray(y_test, dtype=np.float32)

if not np.isfinite(X_train_dnn).all():
    raise ValueError("X_train_dnn còn giá trị NaN hoặc vô cực.")

if not np.isfinite(X_val_dnn).all():
    raise ValueError("X_val_dnn còn giá trị NaN hoặc vô cực.")

if not np.isfinite(X_test_dnn).all():
    raise ValueError("X_test_dnn còn giá trị NaN hoặc vô cực.")

print("X_train_dnn:", X_train_dnn.shape)
print("X_val_dnn  :", X_val_dnn.shape)
print("X_test_dnn :", X_test_dnn.shape)
print("y_train_dnn:", y_train_dnn.shape)
print("y_val_dnn  :", y_val_dnn.shape)
print("y_test_dnn :", y_test_dnn.shape)
print("Số đặc trưng:", NUM_FEATURES)
print("Số lớp:", NUM_CLASSES)
print("Kiểu dữ liệu X:", X_train_dnn.dtype)

In [ ]:
%%time

# =========================================================
# BƯỚC 11: XÂY DỰNG VÀ HUẤN LUYỆN DNN
# =========================================================

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

tf.keras.utils.set_random_seed(42)

# Thư mục lưu model và các thành phần tiền xử lý.
# Mỗi chế độ chia dữ liệu được lưu vào một thư mục con riêng để các lần
# chạy không ghi đè lên nhau, tiện so sánh kết quả về sau.
model_output_dir = os.path.abspath(
    os.path.join("output_after_preprocess", "dnn_model", SPLIT_MODE)
)
os.makedirs(model_output_dir, exist_ok=True)

best_model_file = os.path.join(model_output_dir, "best_dnn_model.keras")
final_model_file = os.path.join(model_output_dir, "final_dnn_model.keras")
history_file = os.path.join(model_output_dir, "training_history.csv")
config_file = os.path.join(model_output_dir, "training_config.json")

EPOCHS = 50
BATCH_SIZE = 4096
LEARNING_RATE = 1e-3

# Chỉ số dùng để chọn model tốt nhất và dừng sớm.
# ROC-AUC bão hòa quá nhanh (chỉ dao động ở chữ số thập phân thứ 5)
# nên việc chọn "epoch tốt nhất" theo nó thực chất là chọn theo nhiễu.
# PR-AUC nhạy hơn nhiều trên tập validation mất cân bằng.
MONITOR_METRIC = "val_pr_auc"
MONITOR_MODE = "max"

# ---------------------------------------------------------
# Xây dựng kiến trúc DNN
# ---------------------------------------------------------

model = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(NUM_FEATURES,),
        name="input_features"
    ),

    tf.keras.layers.Dense(
        256,
        kernel_initializer="he_normal",
        use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dropout(0.30),

    tf.keras.layers.Dense(
        128,
        kernel_initializer="he_normal",
        use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Dense(
        64,
        kernel_initializer="he_normal",
        use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dropout(0.20),

    tf.keras.layers.Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        name="ddos_probability"
    )
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="roc_auc", curve="ROC"),
        tf.keras.metrics.AUC(name="pr_auc", curve="PR")
    ]
)

model.summary()

# ---------------------------------------------------------
# Callbacks
# ---------------------------------------------------------

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=best_model_file,
        monitor=MONITOR_METRIC,
        mode=MONITOR_MODE,
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor=MONITOR_METRIC,
        mode=MONITOR_MODE,
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor=MONITOR_METRIC,
        mode=MONITOR_MODE,
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),

    tf.keras.callbacks.TerminateOnNaN(),

    tf.keras.callbacks.CSVLogger(history_file)
]

# ---------------------------------------------------------
# Train model
# ---------------------------------------------------------
# Dùng validation_data thay cho validation_split để kiểm soát rõ ràng
# tập validation đã tách ở Bước 9 (giữ phân bố nhãn tự nhiên).

history = model.fit(
    X_train_dnn,
    y_train_dnn,
    validation_data=(X_val_dnn, y_val_dnn),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    callbacks=callbacks,
    verbose=1
)

# Lưu model có trọng số tốt nhất do EarlyStopping khôi phục
model.save(final_model_file)

# Lưu cấu hình thí nghiệm để đối chiếu giữa các lần chạy
training_config = {
    "split_mode": SPLIT_MODE,
    "split_description": SPLIT_DESCRIPTION,
    "train_days": TRAIN_DAYS if SPLIT_MODE == "temporal_by_day" else None,
    "test_days": TEST_DAYS if SPLIT_MODE == "temporal_by_day" else None,
    "test_size": TEST_SIZE,
    "drop_dst_port": DROP_DST_PORT,
    "excluded_columns": EXCLUDED_COLUMNS,
    "num_features": int(NUM_FEATURES),
    "feature_names": feature_names,
    "epochs_configured": EPOCHS,
    "epochs_run": len(history.history["loss"]),
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "monitor_metric": MONITOR_METRIC,
    "train_samples": int(len(y_train_dnn)),
    "validation_samples": int(len(y_val_dnn)),
    "test_samples": int(len(y_test_dnn))
}

with open(config_file, "w", encoding="utf-8") as config_handle:
    json.dump(training_config, config_handle, ensure_ascii=False, indent=2)

print(f"\nĐã lưu model tốt nhất: {best_model_file}")
print(f"Đã lưu model cuối cùng: {final_model_file}")
print(f"Đã lưu cấu hình thí nghiệm: {config_file}")

In [ ]:
%%time

# =========================================================
# BƯỚC 12: ĐÁNH GIÁ DNN TRÊN TẬP TEST
# =========================================================

import os
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score
)

class_codes = sorted(inverse_label_mapping)
class_names = [inverse_label_mapping[code] for code in class_codes]

y_true_test = np.asarray(y_test_dnn, dtype=np.int8).ravel()
y_true_val = np.asarray(y_val_dnn, dtype=np.int8).ravel()

# ---------------------------------------------------------
# 1. Dự đoán xác suất
# ---------------------------------------------------------

y_probability_val = model.predict(
    X_val_dnn,
    batch_size=4096,
    verbose=1
).ravel()

y_probability_test = model.predict(
    X_test_dnn,
    batch_size=4096,
    verbose=1
).ravel()

# ---------------------------------------------------------
# 2. Dò ngưỡng quyết định TRÊN TẬP VALIDATION
# ---------------------------------------------------------
# Ngưỡng phải được chọn trên validation rồi mới áp lên test.
# Chọn ngưỡng trực tiếp trên test là một dạng rò rỉ dữ liệu.

precision_curve, recall_curve, threshold_grid = precision_recall_curve(
    y_true_val,
    y_probability_val
)

f1_curve = np.divide(
    2 * precision_curve * recall_curve,
    precision_curve + recall_curve,
    out=np.zeros_like(precision_curve),
    where=(precision_curve + recall_curve) > 0
)

# threshold_grid ngắn hơn precision/recall đúng 1 phần tử.
best_threshold_position = int(np.argmax(f1_curve[:-1]))
best_threshold = float(threshold_grid[best_threshold_position])

print(f"\nNgưỡng mặc định: 0.5")
print(
    f"Ngưỡng tối ưu F1 dò trên validation: {best_threshold:.6f} "
    f"(F1 validation = {f1_curve[best_threshold_position]:.6f})"
)

# ---------------------------------------------------------
# 3. Chỉ số không phụ thuộc ngưỡng
# ---------------------------------------------------------

test_roc_auc = roc_auc_score(y_true_test, y_probability_test)
test_pr_auc = average_precision_score(y_true_test, y_probability_test)

print(f"\nROC-AUC trên test: {test_roc_auc:.6f}")
print(f"PR-AUC  trên test: {test_pr_auc:.6f}")


# ---------------------------------------------------------
# 4. Báo cáo chi tiết tại từng ngưỡng
# ---------------------------------------------------------

def report_at_threshold(threshold, threshold_name):
    """In báo cáo đầy đủ cho một ngưỡng quyết định."""
    y_pred_local = (y_probability_test >= threshold).astype(np.int8)

    cm_local = confusion_matrix(y_true_test, y_pred_local, labels=class_codes)
    true_negative, false_positive, false_negative, true_positive = (
        cm_local.ravel()
    )

    accuracy = (true_positive + true_negative) / cm_local.sum()
    precision = (
        true_positive / (true_positive + false_positive)
        if (true_positive + false_positive) > 0
        else 0.0
    )
    recall = (
        true_positive / (true_positive + false_negative)
        if (true_positive + false_negative) > 0
        else 0.0
    )
    f1_score_value = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )
    false_positive_rate = (
        false_positive / (false_positive + true_negative)
        if (false_positive + true_negative) > 0
        else 0.0
    )

    print("\n=========================================================")
    print(f"NGƯỠNG {threshold_name} = {threshold:.6f}")
    print("=========================================================")
    print(classification_report(
        y_true_test,
        y_pred_local,
        labels=class_codes,
        target_names=class_names,
        digits=6,
        zero_division=0
    ))

    confusion_df_local = pd.DataFrame(
        cm_local,
        index=[f"Thực tế {name}" for name in class_names],
        columns=[f"Dự đoán {name}" for name in class_names]
    )
    display(confusion_df_local)

    print(f"Accuracy            : {accuracy:.6f}")
    print(f"Precision (DDoS)    : {precision:.6f}")
    print(f"Recall (DDoS)       : {recall:.6f}")
    print(f"F1 (DDoS)           : {f1_score_value:.6f}")
    print(
        f"False Positive Rate : {false_positive_rate:.6f} "
        f"({false_positive:,} cảnh báo sai trên {false_positive + true_negative:,} "
        f"flow Benign)"
    )
    print(f"Bỏ sót (FN)         : {false_negative:,} flow DDoS")

    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1_score_value),
        "false_positive_rate": float(false_positive_rate),
        "true_negative": int(true_negative),
        "false_positive": int(false_positive),
        "false_negative": int(false_negative),
        "true_positive": int(true_positive)
    }


metrics_default = report_at_threshold(0.5, "MẶC ĐỊNH")
metrics_tuned = report_at_threshold(best_threshold, "TỐI ƯU F1")

# Nhãn dự đoán dùng cho Bước 13 (theo ngưỡng mặc định).
y_pred = (y_probability_test >= 0.5).astype(np.int8)

# ---------------------------------------------------------
# 5. Kết quả theo từng biến thể tấn công
# ---------------------------------------------------------
# Con số tổng có thể che giấu việc mô hình bỏ sót hoàn toàn một loại
# tấn công hiếm (ví dụ LOIC-UDP chỉ có khoảng 1.700 mẫu).

print("\n=========================================================")
print("KẾT QUẢ THEO TỪNG BIẾN THỂ TẤN CÔNG (ngưỡng 0.5)")
print("=========================================================")

attack_rows = []

for code, attack_name in enumerate(attack_type_names):
    subset_mask = attack_type_codes_test == code
    subset_size = int(subset_mask.sum())

    if subset_size == 0:
        continue

    subset_predictions = y_pred[subset_mask]
    subset_truth = y_true_test[subset_mask]
    is_attack = subset_truth[0] == 1

    if is_attack:
        detected = int(subset_predictions.sum())
        attack_rows.append({
            "Loại": attack_name,
            "Vai trò": "Tấn công",
            "Số mẫu": subset_size,
            "Phân loại đúng": detected,
            "Phân loại sai": subset_size - detected,
            "Tỷ lệ đúng": detected / subset_size,
            "Ý nghĩa": "Recall - phân loại sai nghĩa là bỏ sót tấn công"
        })
    else:
        false_alarms = int(subset_predictions.sum())
        attack_rows.append({
            "Loại": attack_name,
            "Vai trò": "Lưu lượng bình thường",
            "Số mẫu": subset_size,
            "Phân loại đúng": subset_size - false_alarms,
            "Phân loại sai": false_alarms,
            "Tỷ lệ đúng": (subset_size - false_alarms) / subset_size,
            "Ý nghĩa": "Specificity - phân loại sai nghĩa là cảnh báo nhầm"
        })

attack_report_df = pd.DataFrame(attack_rows)

with pd.option_context("display.max_colwidth", None):
    display(attack_report_df)

# ---------------------------------------------------------
# 6. Ước lượng precision khi triển khai thật
# ---------------------------------------------------------
# Tập test giữ phân bố của bộ dữ liệu, nhưng trong mạng thật lưu lượng
# bình thường áp đảo hơn nhiều. Precision phụ thuộc rất mạnh vào tỷ lệ này.

recall_at_default = metrics_default["recall"]
fpr_at_default = metrics_default["false_positive_rate"]

prior_rows = []

for attack_ratio in (0.5, 0.1, 0.01, 0.001, 0.0001):
    numerator = attack_ratio * recall_at_default
    denominator = numerator + (1 - attack_ratio) * fpr_at_default
    estimated_precision = numerator / denominator if denominator > 0 else 0.0

    prior_rows.append({
        "Tỷ lệ DDoS trong lưu lượng": f"{attack_ratio:.4%}",
        "Recall": recall_at_default,
        "FPR": fpr_at_default,
        "Precision ước lượng": estimated_precision
    })

print("\nPrecision ước lượng theo tỷ lệ tấn công thực tế (ngưỡng 0.5):")
display(pd.DataFrame(prior_rows))

# ---------------------------------------------------------
# 7. Lưu toàn bộ kết quả
# ---------------------------------------------------------

evaluation_summary = {
    "split_mode": SPLIT_MODE,
    "test_samples": int(len(y_true_test)),
    "test_benign": int((y_true_test == 0).sum()),
    "test_ddos": int((y_true_test == 1).sum()),
    "roc_auc": float(test_roc_auc),
    "pr_auc": float(test_pr_auc),
    "threshold_default": metrics_default,
    "threshold_tuned_on_validation": metrics_tuned,
    "per_attack_type": attack_report_df.to_dict(orient="records")
}

metrics_file = os.path.join(model_output_dir, "evaluation_metrics.json")

with open(metrics_file, "w", encoding="utf-8") as metrics_handle:
    json.dump(evaluation_summary, metrics_handle, ensure_ascii=False, indent=2)

attack_report_df.to_csv(
    os.path.join(model_output_dir, "per_attack_type_report.csv"),
    index=False,
    encoding="utf-8"
)

print(f"\nĐã lưu kết quả đánh giá tại: {metrics_file}")

In [ ]:
%%time

# =========================================================
# BƯỚC 13: XUẤT CONFUSION MATRIX VÀ CÁC ĐỒ THỊ ĐÁNH GIÁ
# =========================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score
)

model_output_dir = os.path.abspath(
    os.path.join("output_after_preprocess", "dnn_model", SPLIT_MODE)
)
os.makedirs(model_output_dir, exist_ok=True)

confusion_image_file = os.path.join(
    model_output_dir,
    "confusion_matrix_dnn.png"
)
training_image_file = os.path.join(
    model_output_dir,
    "training_validation_curves.png"
)
curves_image_file = os.path.join(
    model_output_dir,
    "roc_pr_curves.png"
)
history_csv_file = os.path.join(model_output_dir, "training_history.csv")
best_model_file = os.path.join(model_output_dir, "best_dnn_model.keras")

# =========================================================
# 1. LẤY XÁC SUẤT DỰ ĐOÁN
# =========================================================

if "X_test_dnn" not in globals() or "y_test_dnn" not in globals():
    raise NameError(
        "Không tìm thấy X_test_dnn hoặc y_test_dnn. "
        "Hãy chạy lại Bước 10 trước."
    )

y_true = np.asarray(y_test_dnn, dtype=np.int8).ravel()

# Tái sử dụng kết quả của Bước 12 nếu có để khỏi dự đoán lại.
if "y_probability_test" in globals():
    print("Dùng lại xác suất dự đoán đã tính ở Bước 12.")
    y_probability = y_probability_test
else:
    if os.path.isfile(best_model_file):
        print(f"Đang tải model tốt nhất: {best_model_file}")
        evaluation_model = tf.keras.models.load_model(best_model_file)
    elif "model" in globals():
        evaluation_model = model
    else:
        raise FileNotFoundError(
            "Không tìm thấy best_dnn_model.keras và biến model "
            "không tồn tại trong bộ nhớ."
        )

    y_probability = evaluation_model.predict(
        X_test_dnn,
        batch_size=4096,
        verbose=1
    ).ravel()

y_pred = (y_probability >= 0.5).astype(np.int8)

if "inverse_label_mapping" in globals():
    plot_inverse_mapping = inverse_label_mapping
else:
    plot_inverse_mapping = {0: "Benign", 1: "DDoS"}

class_codes = sorted(plot_inverse_mapping)
class_names = [plot_inverse_mapping[code] for code in class_codes]

split_note = globals().get("SPLIT_DESCRIPTION", SPLIT_MODE)

# =========================================================
# 2. CONFUSION MATRIX
# =========================================================

cm = confusion_matrix(y_true, y_pred, labels=class_codes)

# Tính phần trăm theo từng nhãn thực tế
cm_percentage = np.divide(
    cm,
    cm.sum(axis=1, keepdims=True),
    out=np.zeros_like(cm, dtype=float),
    where=cm.sum(axis=1, keepdims=True) != 0
) * 100

annotations = np.empty_like(cm, dtype=object)

for row in range(cm.shape[0]):
    for column in range(cm.shape[1]):
        annotations[row, column] = (
            f"{cm[row, column]:,}\n"
            f"{cm_percentage[row, column]:.2f}%"
        )

fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=annotations,
    fmt="",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=True,
    linewidths=0.5,
    ax=ax
)

ax.set_title(
    f"Confusion Matrix - DNN phát hiện DDoS\n({split_note})",
    fontsize=12
)
ax.set_xlabel("Nhãn dự đoán")
ax.set_ylabel("Nhãn thực tế")

plt.tight_layout()
fig.savefig(confusion_image_file, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Đã lưu Confusion Matrix tại:\n{confusion_image_file}")

# =========================================================
# 3. ĐỌC LỊCH SỬ TRAIN
# =========================================================

if "history" in globals() and hasattr(history, "history"):
    history_df = pd.DataFrame(history.history)
    history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
elif os.path.isfile(history_csv_file):
    history_df = pd.read_csv(history_csv_file)

    # CSVLogger lưu epoch từ 0 nên cộng thêm 1
    if "epoch" in history_df.columns:
        history_df["epoch"] = history_df["epoch"] + 1
else:
    raise FileNotFoundError(
        f"Không tìm thấy lịch sử huấn luyện: {history_csv_file}"
    )

epochs = history_df["epoch"].to_numpy()

# =========================================================
# 4. ĐỒ THỊ TRAIN/VALIDATION THEO EPOCH
# =========================================================
# Lưu ý khi đọc đồ thị: tập train đã được cân bằng 50/50 còn tập
# validation giữ phân bố tự nhiên, nên hai đường không so sánh trực
# tiếp được về mặt giá trị tuyệt đối. Điều cần nhìn là XU HƯỚNG:
# validation đi xuống rồi đi ngang là hội tụ, đi lên là quá khớp.

metric_pairs = [
    ("loss", "val_loss", "Loss"),
    ("pr_auc", "val_pr_auc", "PR-AUC"),
    ("recall", "val_recall", "Recall"),
    ("accuracy", "val_accuracy", "Accuracy")
]

available_metrics = [
    metric
    for metric in metric_pairs
    if metric[0] in history_df.columns and metric[1] in history_df.columns
]

if not available_metrics:
    raise KeyError(
        "Không tìm thấy các cột train/validation trong history."
    )

# Epoch tốt nhất theo đúng chỉ số đã dùng để dừng sớm ở Bước 11.
if "val_pr_auc" in history_df.columns:
    best_position = int(np.argmax(history_df["val_pr_auc"].to_numpy()))
    best_metric_name = "val_pr_auc"
else:
    best_position = int(np.argmin(history_df["val_loss"].to_numpy()))
    best_metric_name = "val_loss"

best_epoch = epochs[best_position]

fig, axes = plt.subplots(
    1,
    len(available_metrics),
    figsize=(5.5 * len(available_metrics), 5)
)
axes = np.atleast_1d(axes)

for ax, (train_metric, val_metric, title) in zip(axes, available_metrics):
    ax.plot(
        epochs,
        history_df[train_metric],
        label=f"Train {title}",
        linewidth=2
    )
    ax.plot(
        epochs,
        history_df[val_metric],
        label=f"Validation {title}",
        linewidth=2
    )
    ax.axvline(
        best_epoch,
        color="red",
        linestyle="--",
        alpha=0.7,
        label=f"Best epoch: {best_epoch}"
    )

    ax.set_title(f"{title} theo Epoch")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.grid(True, alpha=0.3)
    ax.legend()

fig.suptitle(
    "Quá trình huấn luyện và validation của DNN\n"
    f"({split_note}; train cân bằng 50/50, validation giữ phân bố tự nhiên)",
    fontsize=13
)

plt.tight_layout()
fig.savefig(training_image_file, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Epoch tốt nhất (theo {best_metric_name}): {best_epoch}")
print(f"Đã lưu biểu đồ train/validation tại:\n{training_image_file}")

# =========================================================
# 5. ĐƯỜNG CONG ROC VÀ PRECISION-RECALL TRÊN TẬP TEST
# =========================================================
# Với dữ liệu mất cân bằng, đường Precision-Recall phản ánh chất lượng
# mô hình tốt hơn ROC vì ROC gần như luôn đẹp khi lớp âm quá đông.

false_positive_rate, true_positive_rate, _ = roc_curve(y_true, y_probability)
precision_values, recall_values, _ = precision_recall_curve(
    y_true,
    y_probability
)

roc_auc_value = roc_auc_score(y_true, y_probability)
pr_auc_value = average_precision_score(y_true, y_probability)
positive_ratio = float((y_true == 1).mean())

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

axes[0].plot(
    false_positive_rate,
    true_positive_rate,
    linewidth=2,
    label=f"DNN (AUC = {roc_auc_value:.6f})"
)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Đoán ngẫu nhiên")
axes[0].set_title("Đường cong ROC trên tập test")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="lower right")

axes[1].plot(
    recall_values,
    precision_values,
    linewidth=2,
    label=f"DNN (PR-AUC = {pr_auc_value:.6f})"
)
axes[1].axhline(
    positive_ratio,
    color="k",
    linestyle="--",
    alpha=0.5,
    label=f"Mức cơ sở = {positive_ratio:.4f}"
)
axes[1].set_title("Đường cong Precision-Recall trên tập test")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="lower left")

fig.suptitle(f"Đánh giá không phụ thuộc ngưỡng\n({split_note})", fontsize=13)

plt.tight_layout()
fig.savefig(curves_image_file, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Đã lưu đường cong ROC/PR tại:\n{curves_image_file}")

In [ ]:
%%time

# =========================================================
# BƯỚC 14: ĐÁNH GIÁ ĐỘ ẢNH HƯỞNG CỦA TỪNG ĐẶC TRƯNG
# =========================================================
# Dùng hai phương pháp bổ trợ nhau:
#
# 1. Permutation importance (kết quả chính)
#    Xáo trộn ngẫu nhiên giá trị của MỘT cột trên tập test rồi đo xem
#    chỉ số đánh giá tụt bao nhiêu. Tụt càng nhiều nghĩa là mô hình
#    càng phụ thuộc vào cột đó. Phương pháp này không phụ thuộc kiến
#    trúc mô hình nên dùng làm con số đưa vào báo cáo.
#
# 2. Gradient saliency (đối chiếu)
#    Đo trung bình |đạo hàm của xác suất đầu ra theo từng đặc trưng|.
#    Rẻ hơn nhiều, cho biết mô hình nhạy với cột nào ở mức cục bộ.
#    Dùng để kiểm tra chéo: hai bảng xếp hạng nên đồng thuận với nhau.
#
# Bảng kết quả có hai loại phần trăm, KHÔNG được nhầm lẫn:
#   - "Tỷ lệ đóng góp (%)": phần của đặc trưng này trong TỔNG mức ảnh
#     hưởng của toàn bộ đặc trưng. Cả cột cộng lại bằng 100%. Đây chính
#     là tỷ lệ phụ thuộc của mô hình vào từng đặc trưng.
#   - "Sụt giảm so với baseline (%)": xáo trộn riêng đặc trưng này thì
#     chỉ số đánh giá mất bao nhiêu phần trăm so với ban đầu. Cột này
#     KHÔNG cộng lại thành 100%.
#
# LƯU Ý khi đọc kết quả: hai đặc trưng tương quan mạnh sẽ CHIA NHAU độ
# quan trọng và có thể cùng hiện ra là "không quan trọng", vì khi xáo
# trộn cột này mô hình vẫn còn cột kia để dựa vào. Điểm thấp KHÔNG
# đồng nghĩa với vô dụng.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import roc_auc_score, average_precision_score

# ---------------------------------------------------------
# Cấu hình
# ---------------------------------------------------------

# Số mẫu lấy từ tập test để đo. Giảm xuống nếu muốn chạy nhanh hơn.
IMPORTANCE_SAMPLE_SIZE = 200_000

# Số lần xáo trộn lặp lại cho mỗi đặc trưng (để có độ lệch chuẩn).
IMPORTANCE_REPEATS = 3

# Chỉ số dùng để đo mức sụt giảm: 'pr_auc' hoặc 'roc_auc'.
# PR-AUC phù hợp hơn với dữ liệu mất cân bằng.
IMPORTANCE_METRIC = "pr_auc"

# Số đặc trưng hiển thị trên biểu đồ.
IMPORTANCE_TOP_K = 20

# Các mốc đóng góp lũy kế cần báo cáo.
IMPORTANCE_COVERAGE_TARGETS = (50, 80, 90, 95, 99)

# Đặt False nếu muốn bỏ qua phần gradient cho nhanh.
RUN_GRADIENT_IMPORTANCE = True

IMPORTANCE_RANDOM_STATE = 42

model_output_dir = os.path.abspath(
    os.path.join("output_after_preprocess", "dnn_model", SPLIT_MODE)
)
os.makedirs(model_output_dir, exist_ok=True)

importance_csv_file = os.path.join(
    model_output_dir,
    "feature_importance.csv"
)
importance_image_file = os.path.join(
    model_output_dir,
    "feature_importance.png"
)
contribution_image_file = os.path.join(
    model_output_dir,
    "feature_contribution_all.png"
)

best_model_file = os.path.join(model_output_dir, "best_dnn_model.keras")

if "model" in globals():
    importance_model = model
elif os.path.isfile(best_model_file):
    print(f"Đang tải model tốt nhất: {best_model_file}")
    importance_model = tf.keras.models.load_model(best_model_file)
else:
    raise FileNotFoundError(
        "Không tìm thấy model để đánh giá đặc trưng. Hãy chạy lại Bước 11."
    )

if len(feature_names) != X_test_dnn.shape[1]:
    raise ValueError(
        f"Số feature không khớp: feature_names={len(feature_names)}, "
        f"X_test_dnn={X_test_dnn.shape[1]}"
    )

# ---------------------------------------------------------
# 1. Lấy mẫu con của tập test (giữ nguyên tỷ lệ hai lớp)
# ---------------------------------------------------------

y_true_importance = np.asarray(y_test_dnn, dtype=np.int8).ravel()
importance_rng = np.random.default_rng(IMPORTANCE_RANDOM_STATE)

if IMPORTANCE_SAMPLE_SIZE >= len(y_true_importance):
    sample_index = np.arange(len(y_true_importance))
    print(f"Dùng toàn bộ {len(sample_index):,} mẫu test.")
else:
    sample_blocks = []
    sampling_ratio = IMPORTANCE_SAMPLE_SIZE / len(y_true_importance)

    for class_code in (0, 1):
        class_positions = np.flatnonzero(y_true_importance == class_code)
        take = max(1, int(round(len(class_positions) * sampling_ratio)))
        sample_blocks.append(
            importance_rng.choice(class_positions, size=take, replace=False)
        )

    sample_index = np.sort(np.concatenate(sample_blocks))
    print(
        f"Lấy mẫu {len(sample_index):,} / {len(y_true_importance):,} mẫu test "
        f"(giữ nguyên tỷ lệ Benign/DDoS)."
    )

X_importance = np.array(X_test_dnn[sample_index], dtype=np.float32, copy=True)
y_importance = y_true_importance[sample_index]

print(
    f"  Benign: {(y_importance == 0).sum():,} | "
    f"DDoS: {(y_importance == 1).sum():,}"
)


def predict_probability(matrix):
    """Dự đoán xác suất DDoS cho một ma trận đặc trưng."""
    return importance_model.predict(
        matrix,
        batch_size=8192,
        verbose=0
    ).ravel()


def score_probability(labels, probability):
    """Tính chỉ số đánh giá đã chọn."""
    if IMPORTANCE_METRIC == "roc_auc":
        return float(roc_auc_score(labels, probability))
    return float(average_precision_score(labels, probability))


# ---------------------------------------------------------
# 2. Permutation importance
# ---------------------------------------------------------

baseline_score = score_probability(
    y_importance,
    predict_probability(X_importance)
)

print(
    f"\n{IMPORTANCE_METRIC.upper()} gốc (chưa xáo trộn): "
    f"{baseline_score:.6f}"
)
print(
    f"Bắt đầu đo {len(feature_names)} đặc trưng "
    f"x {IMPORTANCE_REPEATS} lần xáo trộn..."
)

permutation_rows = []

for position, feature_name in enumerate(feature_names):
    original_column = X_importance[:, position].copy()
    permuted_scores = []

    for repeat in range(IMPORTANCE_REPEATS):
        X_importance[:, position] = importance_rng.permutation(original_column)
        permuted_scores.append(
            score_probability(
                y_importance,
                predict_probability(X_importance)
            )
        )

    # Trả lại giá trị gốc trước khi sang đặc trưng tiếp theo
    X_importance[:, position] = original_column

    permuted_scores = np.asarray(permuted_scores, dtype=np.float64)
    mean_drop = baseline_score - permuted_scores.mean()

    permutation_rows.append({
        "Feature": feature_name,
        "Mức sụt giảm trung bình": mean_drop,
        "Độ lệch chuẩn": float(permuted_scores.std()),
        f"{IMPORTANCE_METRIC} sau khi xáo trộn": float(permuted_scores.mean()),
        "Sụt giảm so với baseline (%)": (
            mean_drop / baseline_score * 100 if baseline_score > 0 else 0.0
        )
    })

    if (position + 1) % 10 == 0 or position + 1 == len(feature_names):
        print(f"  Đã đo {position + 1}/{len(feature_names)} đặc trưng")

importance_df = pd.DataFrame(permutation_rows).sort_values(
    "Mức sụt giảm trung bình",
    ascending=False
).reset_index(drop=True)

# ---------------------------------------------------------
# 3. Tỷ lệ phụ thuộc của mô hình vào từng đặc trưng
# ---------------------------------------------------------
# Chuẩn hóa mức sụt giảm về tổng 100% để trả lời trực tiếp câu hỏi
# "mô hình phụ thuộc bao nhiêu phần trăm vào đặc trưng nào".
# Mức sụt giảm âm (xáo trộn xong lại tốt hơn) là nhiễu đo đạc, quy về 0.

positive_drop = importance_df["Mức sụt giảm trung bình"].clip(lower=0)
total_positive_drop = float(positive_drop.sum())

if total_positive_drop > 0:
    importance_df["Tỷ lệ đóng góp (%)"] = (
        positive_drop / total_positive_drop * 100
    )
else:
    importance_df["Tỷ lệ đóng góp (%)"] = 0.0
    print(
        "\nCẢNH BÁO: không đặc trưng nào làm chỉ số sụt giảm. "
        "Kiểm tra lại model hoặc tăng IMPORTANCE_SAMPLE_SIZE."
    )

importance_df["Đóng góp lũy kế (%)"] = (
    importance_df["Tỷ lệ đóng góp (%)"].cumsum()
)

importance_df.insert(0, "Hạng", np.arange(1, len(importance_df) + 1))

# ---------------------------------------------------------
# 4. Gradient saliency (đối chiếu)
# ---------------------------------------------------------

if RUN_GRADIENT_IMPORTANCE:
    print("\nĐang tính gradient saliency...")

    gradient_sum = np.zeros(len(feature_names), dtype=np.float64)
    gradient_batch_size = 8192

    for start in range(0, len(X_importance), gradient_batch_size):
        batch = tf.convert_to_tensor(
            X_importance[start:start + gradient_batch_size]
        )

        with tf.GradientTape() as tape:
            tape.watch(batch)
            predictions = importance_model(batch, training=False)

        batch_gradients = tape.gradient(predictions, batch)
        gradient_sum += np.abs(batch_gradients.numpy()).sum(axis=0)

    gradient_mean = gradient_sum / len(X_importance)
    gradient_total = float(gradient_mean.sum())

    gradient_df = pd.DataFrame({
        "Feature": feature_names,
        "Gradient trung bình": gradient_mean,
        "Tỷ lệ gradient (%)": (
            gradient_mean / gradient_total * 100
            if gradient_total > 0
            else np.zeros_like(gradient_mean)
        )
    })

    importance_df = importance_df.merge(gradient_df, on="Feature", how="left")

    # Hệ số tương quan hạng Spearman giữa hai phương pháp.
    # Tính bằng Pearson trên thứ hạng để không cần thêm thư viện.
    rank_permutation = importance_df["Mức sụt giảm trung bình"].rank()
    rank_gradient = importance_df["Gradient trung bình"].rank()
    rank_correlation = float(rank_permutation.corr(rank_gradient))

    print(
        f"Tương quan hạng giữa permutation và gradient: "
        f"{rank_correlation:.4f}"
    )
    print(
        "  (gần 1 nghĩa là hai phương pháp đồng thuận; "
        "gần 0 nghĩa là kết quả chưa đáng tin, nên tăng IMPORTANCE_REPEATS)"
    )

# Sắp xếp lại cột cho dễ đọc: tỷ lệ phụ thuộc lên trước.
column_order = [
    "Hạng",
    "Feature",
    "Tỷ lệ đóng góp (%)",
    "Đóng góp lũy kế (%)",
    "Mức sụt giảm trung bình",
    "Độ lệch chuẩn",
    "Sụt giảm so với baseline (%)",
    f"{IMPORTANCE_METRIC} sau khi xáo trộn",
]

if RUN_GRADIENT_IMPORTANCE:
    column_order += ["Gradient trung bình", "Tỷ lệ gradient (%)"]

importance_df = importance_df[column_order]

# ---------------------------------------------------------
# 5. Bảng kết quả
# ---------------------------------------------------------

print("\n=========================================================")
print(f"TỶ LỆ PHỤ THUỘC CỦA MÔ HÌNH VÀO TỪNG ĐẶC TRƯNG")
print(f"({SPLIT_DESCRIPTION})")
print("=========================================================")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.float_format", "{:.6f}".format
):
    display(importance_df)

# --- Mức độ tập trung: cần bao nhiêu đặc trưng để đạt X% ảnh hưởng ---

print("\nSố đặc trưng cần thiết để đạt các mốc đóng góp lũy kế:")

coverage_rows = []
cumulative_values = importance_df["Đóng góp lũy kế (%)"].to_numpy()

for target in IMPORTANCE_COVERAGE_TARGETS:
    reached = np.flatnonzero(cumulative_values >= target)
    needed = int(reached[0]) + 1 if len(reached) > 0 else len(importance_df)

    coverage_rows.append({
        "Mốc đóng góp lũy kế (%)": target,
        "Số đặc trưng cần": needed,
        "Tỷ lệ trên tổng đặc trưng (%)": needed / len(importance_df) * 100
    })

    print(
        f"  {target:>3}% ảnh hưởng <- {needed:>3} / {len(importance_df)} "
        f"đặc trưng ({needed / len(importance_df) * 100:.1f}%)"
    )

display(pd.DataFrame(coverage_rows))

# --- Nhóm đặc trưng không đóng góp ---

useless_features = importance_df.loc[
    importance_df["Mức sụt giảm trung bình"] <= 0,
    "Feature"
].tolist()

print(
    f"\nSố đặc trưng gần như không đóng góp (mức sụt giảm <= 0): "
    f"{len(useless_features)} / {len(importance_df)}"
)

if useless_features:
    print("  " + ", ".join(useless_features))
    print(
        "  Lưu ý: có thể do tương quan với đặc trưng khác chứ không hẳn là "
        "vô dụng. Muốn kết luận chắc chắn thì train lại sau khi bỏ chúng "
        "và so sánh kết quả."
    )

shown_top_k = min(IMPORTANCE_TOP_K, len(importance_df))
top_share = importance_df.head(shown_top_k)["Tỷ lệ đóng góp (%)"].sum()
print(
    f"\n{shown_top_k} đặc trưng đầu bảng chiếm {top_share:.2f}% "
    f"tổng mức ảnh hưởng."
)

importance_df.to_csv(importance_csv_file, index=False, encoding="utf-8")
print(f"Đã lưu bảng độ ảnh hưởng tại:\n{importance_csv_file}")

# ---------------------------------------------------------
# 6. Biểu đồ xếp hạng
# ---------------------------------------------------------

plot_data = importance_df.head(IMPORTANCE_TOP_K).iloc[::-1]

number_of_panels = 2 if RUN_GRADIENT_IMPORTANCE else 1
fig, axes = plt.subplots(
    1,
    number_of_panels,
    figsize=(9 * number_of_panels, max(6, 0.42 * len(plot_data) + 2))
)
axes = np.atleast_1d(axes)

bars = axes[0].barh(
    plot_data["Feature"],
    plot_data["Tỷ lệ đóng góp (%)"],
    xerr=(
        plot_data["Độ lệch chuẩn"] / total_positive_drop * 100
        if total_positive_drop > 0
        else None
    ),
    color="#1f77b4",
    capsize=3
)

for bar, value in zip(bars, plot_data["Tỷ lệ đóng góp (%)"]):
    axes[0].text(
        bar.get_width(),
        bar.get_y() + bar.get_height() / 2,
        f"  {value:.2f}%",
        va="center",
        fontsize=9
    )

axes[0].set_title(
    f"Tỷ lệ phụ thuộc - Top {IMPORTANCE_TOP_K}\n"
    f"(permutation importance trên {IMPORTANCE_METRIC.upper()})",
    fontsize=12
)
axes[0].set_xlabel("Tỷ lệ đóng góp (%)")
axes[0].margins(x=0.15)
axes[0].grid(True, axis="x", alpha=0.3)

if RUN_GRADIENT_IMPORTANCE:
    gradient_plot_data = importance_df.nlargest(
        IMPORTANCE_TOP_K,
        "Tỷ lệ gradient (%)"
    ).iloc[::-1]

    axes[1].barh(
        gradient_plot_data["Feature"],
        gradient_plot_data["Tỷ lệ gradient (%)"],
        color="#ff7f0e"
    )
    axes[1].set_title(
        f"Tỷ lệ gradient saliency - Top {IMPORTANCE_TOP_K}\n"
        "(độ nhạy trung bình của xác suất đầu ra)",
        fontsize=12
    )
    axes[1].set_xlabel("Tỷ lệ gradient (%)")
    axes[1].grid(True, axis="x", alpha=0.3)

fig.suptitle(
    f"Độ ảnh hưởng của đặc trưng - DNN phát hiện DDoS\n({SPLIT_DESCRIPTION})",
    fontsize=13
)

plt.tight_layout()
fig.savefig(importance_image_file, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Đã lưu biểu đồ độ ảnh hưởng tại:\n{importance_image_file}")

# ---------------------------------------------------------
# 7. Biểu đồ đầy đủ - giữ nguyên thứ tự cột trong dataset
# ---------------------------------------------------------
# Hiển thị TOÀN BỘ đặc trưng theo đúng thứ tự cột của dataset (không
# sắp xếp theo độ lớn) để đối chiếu trực tiếp với bảng đặc trưng gốc.

# Sắp lại theo thứ tự trong feature_names, tức thứ tự cột của dataset.
contribution_by_dataset_order = (
    importance_df
    .set_index("Feature")
    .loc[feature_names]
    .reset_index()
)

fig, ax = plt.subplots(
    figsize=(max(12, 0.40 * len(contribution_by_dataset_order)), 7)
)

positions = np.arange(len(contribution_by_dataset_order))
contribution_values = contribution_by_dataset_order["Tỷ lệ đóng góp (%)"]

bar_containers = ax.bar(
    positions,
    contribution_values,
    color="#1f77b4",
    label="Tỷ lệ đóng góp của từng đặc trưng"
)

# Ghi số phần trăm lên đầu mỗi cột, để nằm ngang cho dễ đọc.
ax.bar_label(
    bar_containers,
    labels=[f"{value:.2f}%" for value in contribution_values],
    padding=2,
    fontsize=6
)

# Chừa thêm khoảng trống phía trên để nhãn của cột cao nhất không bị cắt.
ax.set_ylim(0, float(contribution_values.max()) * 1.08)

# Đường mức trung bình: đặc trưng nào nằm trên đường này thì đóng góp
# nhiều hơn mức chia đều. Hữu ích khi biểu đồ không sắp xếp theo độ lớn.
average_contribution = 100 / len(contribution_by_dataset_order)
ax.axhline(
    average_contribution,
    color="#d62728",
    linestyle="--",
    linewidth=1.5,
    alpha=0.8,
    label=f"Mức chia đều ({average_contribution:.2f}%)"
)

ax.set_xticks(positions)
ax.set_xticklabels(
    contribution_by_dataset_order["Feature"],
    rotation=90,
    fontsize=8
)
ax.set_xlim(-0.8, len(contribution_by_dataset_order) - 0.2)
ax.set_ylabel("Tỷ lệ đóng góp (%)")
ax.set_xlabel("Đặc trưng (theo thứ tự cột trong dataset)")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(loc="upper right")

ax.set_title(
    f"Tỷ lệ đóng góp của toàn bộ {len(contribution_by_dataset_order)} đặc trưng "
    "- theo thứ tự trong dataset\n"
    f"({SPLIT_DESCRIPTION})",
    fontsize=13
)

plt.tight_layout()
fig.savefig(contribution_image_file, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

above_average_count = int((contribution_values > average_contribution).sum())
print(
    f"\nSố đặc trưng đóng góp trên mức chia đều "
    f"({average_contribution:.2f}%): {above_average_count} / "
    f"{len(contribution_by_dataset_order)}"
)
print(f"Đã lưu biểu đồ đóng góp đầy đủ tại:\n{contribution_image_file}")

In [ ]:
%%time

# =========================================================
# BƯỚC 15: XUẤT MODEL SANG ĐỊNH DẠNG ONNX
# =========================================================
# ONNX cho phép chạy mô hình bằng onnxruntime (C++/C#/Java/Go/Python)
# mà không cần cài TensorFlow, tốc độ suy luận trên CPU thường nhanh hơn.
#
# Yêu cầu cài đặt (chạy một lần):
#     pip install onnx onnxruntime tf2onnx
#
# Hai chế độ xuất, chọn bằng EXPORT_WITH_SCALER:
#
#   True  - Nhúng luôn StandardScaler vào graph. File ONNX nhận ĐẶC TRƯNG
#           THÔ đúng như trích xuất từ CICFlowMeter. Đây là lựa chọn nên
#           dùng khi triển khai: chỉ cần một file duy nhất, không sợ áp
#           sai tham số chuẩn hóa ở phía ứng dụng.
#
#   False - Chỉ xuất phần mạng nơ-ron. File ONNX nhận đặc trưng ĐÃ được
#           chuẩn hóa sẵn, phía ứng dụng phải tự nạp standard_scaler.joblib
#           và áp dụng trước khi gọi model.

import json
import os
import shutil

import numpy as np
import tensorflow as tf

EXPORT_WITH_SCALER = True
ONNX_OPSET = 17
ONNX_INPUT_NAME = "input"
ONNX_OUTPUT_NAME = "output"

# Số mẫu lấy từ tập test để đối chiếu kết quả ONNX với Keras.
ONNX_VERIFY_SAMPLE_SIZE = 5_000

# Ngưỡng sai lệch tối đa chấp nhận được giữa hai bản.
ONNX_TOLERANCE = 1e-4

# ---------------------------------------------------------
# 1. Kiểm tra thư viện
# ---------------------------------------------------------

try:
    import onnx
    import onnxruntime as ort
except ImportError as import_error:
    raise ImportError(
        "Thiếu thư viện để xuất ONNX. Chạy lệnh sau rồi thực thi lại cell:\n"
        "    pip install onnx onnxruntime tf2onnx\n"
        f"Chi tiết: {import_error}"
    ) from import_error

# Thư mục chứa kết quả huấn luyện (nơi đọc model và scaler ra).
model_output_dir = os.path.abspath(
    os.path.join("output_after_preprocess", "dnn_model", SPLIT_MODE)
)

# Thư mục riêng cho các file đem đi triển khai, tách khỏi artefact huấn luyện.
onnx_output_dir = os.path.join(model_output_dir, "export", "onnx")
os.makedirs(onnx_output_dir, exist_ok=True)

onnx_model_file = os.path.join(onnx_output_dir, "ddos_dnn_model.onnx")
onnx_info_file = os.path.join(onnx_output_dir, "onnx_deployment_info.json")
best_model_file = os.path.join(model_output_dir, "best_dnn_model.keras")

if "model" in globals():
    base_model = model
elif os.path.isfile(best_model_file):
    print(f"Đang tải model tốt nhất: {best_model_file}")
    base_model = tf.keras.models.load_model(best_model_file)
else:
    raise FileNotFoundError(
        "Không tìm thấy model để xuất. Hãy chạy lại Bước 11."
    )

# Scaler cần cho cả việc nhúng vào graph lẫn việc tái tạo đặc trưng thô.
if "scaler" not in globals():
    import joblib

    scaler_file = os.path.join(model_output_dir, "standard_scaler.joblib")
    if not os.path.isfile(scaler_file):
        raise FileNotFoundError(
            f"Không tìm thấy scaler: {scaler_file}. Hãy chạy lại Bước 8."
        )
    scaler = joblib.load(scaler_file)
    print(f"Đã nạp lại scaler từ: {scaler_file}")

scaler_mean = np.asarray(scaler.mean_, dtype=np.float32)
scaler_scale = np.asarray(scaler.scale_, dtype=np.float32)
number_of_features = len(feature_names)

if len(scaler_mean) != number_of_features:
    raise ValueError(
        f"Scaler có {len(scaler_mean)} feature nhưng feature_names có "
        f"{number_of_features}. Hai thứ phải khớp nhau."
    )

# ---------------------------------------------------------
# 2. Dựng model sẽ đem xuất
# ---------------------------------------------------------

if EXPORT_WITH_SCALER:
    # Rescaling lưu hệ số dưới dạng hằng số nên chúng được đóng băng vào
    # graph ONNX. Nếu dùng lớp Normalization thì mean/variance là biến và
    # sẽ trở thành input thừa của file ONNX.
    #   z = (x - mean) / scale  =  x * (1/scale) + (-mean/scale)
    raw_inputs = tf.keras.Input(
        shape=(number_of_features,),
        name=ONNX_INPUT_NAME
    )
    normalized = tf.keras.layers.Rescaling(
        scale=(1.0 / scaler_scale),
        offset=(-scaler_mean / scaler_scale),
        name="standard_scaler"
    )(raw_inputs)

    export_model = tf.keras.Model(
        inputs=raw_inputs,
        outputs=base_model(normalized),
        name="ddos_detector"
    )

    print("Chế độ xuất: ONNX nhận ĐẶC TRƯNG THÔ (đã nhúng sẵn scaler).")
else:
    export_model = base_model
    print("Chế độ xuất: ONNX nhận đặc trưng ĐÃ CHUẨN HÓA.")

print(f"Số đặc trưng đầu vào: {number_of_features}")

# Keras yêu cầu model phải được gọi ít nhất một lần trước khi export,
# nếu không sẽ báo "The model provided has never called".
export_model.predict(
    np.zeros((1, number_of_features), dtype=np.float32),
    verbose=0
)

# ---------------------------------------------------------
# 3. Xuất file ONNX
# ---------------------------------------------------------

input_signature = [
    tf.TensorSpec(
        [None, number_of_features],
        tf.float32,
        name=ONNX_INPUT_NAME
    )
]

export_method = None

try:
    # Cách chính: Keras 3 hỗ trợ sẵn format="onnx".
    export_model.export(onnx_model_file, format="onnx", verbose=False)
    export_method = "keras.Model.export(format='onnx')"
except TypeError:
    export_model.export(onnx_model_file, format="onnx")
    export_method = "keras.Model.export(format='onnx')"
except Exception as keras_export_error:
    print(f"Cách 1 thất bại: {keras_export_error}")
    print("Chuyển sang cách 2: SavedModel + tf2onnx...")

    import tf2onnx

    saved_model_dir = os.path.join(onnx_output_dir, "_saved_model_tmp")
    if os.path.isdir(saved_model_dir):
        shutil.rmtree(saved_model_dir)

    export_model.export(saved_model_dir)
    loaded_saved_model = tf.saved_model.load(saved_model_dir)
    serving_function = loaded_saved_model.signatures["serving_default"]

    @tf.function(input_signature=input_signature)
    def forward(features):
        return list(serving_function(features).values())[0]

    tf2onnx.convert.from_function(
        forward,
        input_signature=input_signature,
        opset=ONNX_OPSET,
        output_path=onnx_model_file
    )

    shutil.rmtree(saved_model_dir, ignore_errors=True)
    export_method = "SavedModel + tf2onnx.convert.from_function"

print(f"\nĐã xuất bằng: {export_method}")
print(f"File ONNX: {onnx_model_file}")
print(f"Dung lượng: {os.path.getsize(onnx_model_file) / 1024:.1f} KB")

# ---------------------------------------------------------
# 4. Kiểm tra tính hợp lệ của file ONNX
# ---------------------------------------------------------

onnx_model = onnx.load(onnx_model_file)

# Đổi tên giá trị giao tiếp ngay trong graph, không chỉ đổi metadata bên ngoài.
def rename_onnx_graph_value(graph, old_name, new_name):
    """Đổi tên một tensor và mọi tham chiếu đến tensor đó trong graph ONNX."""
    for value_collection in (graph.input, graph.output, graph.value_info):
        for value_info in value_collection:
            if value_info.name == old_name:
                value_info.name = new_name

    for initializer in graph.initializer:
        if initializer.name == old_name:
            initializer.name = new_name

    for node in graph.node:
        for position, value_name in enumerate(node.input):
            if value_name == old_name:
                node.input[position] = new_name
        for position, value_name in enumerate(node.output):
            if value_name == old_name:
                node.output[position] = new_name


if len(onnx_model.graph.input) != 1 or len(onnx_model.graph.output) != 1:
    raise RuntimeError(
        "Chỉ hỗ trợ model có đúng 1 input và 1 output để đặt tên cố định."
    )

old_input_name = onnx_model.graph.input[0].name
old_output_name = onnx_model.graph.output[0].name
rename_onnx_graph_value(onnx_model.graph, old_input_name, ONNX_INPUT_NAME)
rename_onnx_graph_value(onnx_model.graph, old_output_name, ONNX_OUTPUT_NAME)
onnx.save(onnx_model, onnx_model_file)

# Nạp lại chính file đã ghi để checker và ONNX Runtime kiểm tra tên thật.
onnx_model = onnx.load(onnx_model_file)
onnx.checker.check_model(onnx_model)
print("onnx.checker: file hợp lệ.")

session = ort.InferenceSession(
    onnx_model_file,
    providers=["CPUExecutionProvider"]
)

session_inputs = session.get_inputs()
session_outputs = session.get_outputs()

if len(session_inputs) != 1:
    raise RuntimeError(
        f"File ONNX có {len(session_inputs)} input, đáng lẽ chỉ có 1. "
        "Nhiều khả năng tham số chuẩn hóa chưa được đóng băng thành hằng số."
    )

onnx_input_name = session_inputs[0].name
onnx_output_name = session_outputs[0].name

if onnx_input_name != ONNX_INPUT_NAME or onnx_output_name != ONNX_OUTPUT_NAME:
    raise RuntimeError(
        "Tên tensor ONNX không đúng sau khi xuất: "
        f"input='{onnx_input_name}', output='{onnx_output_name}'."
    )

print(f"Input : {onnx_input_name} {session_inputs[0].shape}")
print(f"Output: {onnx_output_name} {session_outputs[0].shape}")

# ---------------------------------------------------------
# 5. Đối chiếu kết quả ONNX với Keras trên dữ liệu test
# ---------------------------------------------------------

verify_size = min(ONNX_VERIFY_SAMPLE_SIZE, len(X_test_dnn))
verify_rng = np.random.default_rng(42)
verify_index = verify_rng.choice(len(X_test_dnn), size=verify_size, replace=False)

scaled_features = np.ascontiguousarray(
    X_test_dnn[verify_index],
    dtype=np.float32
)

if EXPORT_WITH_SCALER:
    # Tái tạo đặc trưng thô từ đặc trưng đã chuẩn hóa, rồi chuẩn hóa lại
    # bằng đúng công thức để hai phía cùng xuất phát từ một điểm.
    raw_features = scaled_features * scaler_scale + scaler_mean
    onnx_input_values = raw_features
    keras_input_values = (raw_features - scaler_mean) / scaler_scale
else:
    onnx_input_values = scaled_features
    keras_input_values = scaled_features

keras_probability = base_model.predict(
    keras_input_values,
    batch_size=4096,
    verbose=0
).ravel()

onnx_probability = session.run(
    None,
    {onnx_input_name: np.ascontiguousarray(onnx_input_values, dtype=np.float32)}
)[0].ravel()

max_absolute_difference = float(np.abs(onnx_probability - keras_probability).max())
mean_absolute_difference = float(np.abs(onnx_probability - keras_probability).mean())

print(f"\nĐối chiếu trên {verify_size:,} mẫu test:")
print(f"  Sai lệch xác suất lớn nhất  : {max_absolute_difference:.3e}")
print(f"  Sai lệch xác suất trung bình: {mean_absolute_difference:.3e}")

# Quan trọng hơn sai số xác suất: nhãn dự đoán có giống nhau không.
thresholds_to_check = {"0.5 (mặc định)": 0.5}

if "best_threshold" in globals():
    thresholds_to_check["tối ưu F1 (Bước 12)"] = float(best_threshold)

for threshold_name, threshold_value in thresholds_to_check.items():
    keras_labels = (keras_probability >= threshold_value).astype(np.int8)
    onnx_labels = (onnx_probability >= threshold_value).astype(np.int8)
    disagreement = int((keras_labels != onnx_labels).sum())

    print(
        f"  Ngưỡng {threshold_name}: {disagreement:,} / {verify_size:,} "
        f"mẫu lệch nhãn"
    )

if max_absolute_difference > ONNX_TOLERANCE:
    raise RuntimeError(
        f"Sai lệch {max_absolute_difference:.3e} vượt ngưỡng cho phép "
        f"{ONNX_TOLERANCE:.0e}. KHÔNG dùng file ONNX này."
    )

print("\nKết quả ONNX khớp với Keras trong sai số cho phép.")

# ---------------------------------------------------------
# 6. Lưu thông tin triển khai
# ---------------------------------------------------------
# Thứ tự cột đầu vào PHẢI đúng như feature_names, nếu sai thứ tự thì mô
# hình vẫn chạy nhưng cho kết quả rác.

deployment_info = {
    "onnx_file": os.path.basename(onnx_model_file),
    "export_method": export_method,
    "opset": ONNX_OPSET,
    "includes_scaler": bool(EXPORT_WITH_SCALER),
    "input_name": onnx_input_name,
    "input_shape": ["batch", number_of_features],
    "input_dtype": "float32",
    "input_description": (
        "Đặc trưng THÔ theo đúng thứ tự feature_names"
        if EXPORT_WITH_SCALER
        else "Đặc trưng ĐÃ chuẩn hóa bằng standard_scaler.joblib"
    ),
    "output_name": onnx_output_name,
    "output_description": "Xác suất là DDoS, khoảng [0, 1]",
    "num_features": number_of_features,
    "feature_names": feature_names,
    "label_mapping": {name: int(code) for name, code in label_mapping.items()},
    "split_mode": SPLIT_MODE,
    "split_description": SPLIT_DESCRIPTION,
    "threshold_default": 0.5,
    "threshold_tuned": (
        float(best_threshold) if "best_threshold" in globals() else None
    ),
    "verification": {
        "sample_size": int(verify_size),
        "max_absolute_difference": max_absolute_difference,
        "mean_absolute_difference": mean_absolute_difference,
    },
    "versions": {
        "tensorflow": tf.__version__,
        "keras": tf.keras.__version__,
        "onnx": onnx.__version__,
        "onnxruntime": ort.__version__,
    },
}

with open(onnx_info_file, "w", encoding="utf-8") as info_handle:
    json.dump(deployment_info, info_handle, ensure_ascii=False, indent=2)

print(f"Thư mục xuất ONNX: {onnx_output_dir}")
print(f"Đã lưu thông tin triển khai: {onnx_info_file}")

# ---------------------------------------------------------
# 7. Mẫu code sử dụng
# ---------------------------------------------------------

preprocessing_note = (
    "# File ONNX đã nhúng sẵn scaler: đưa thẳng đặc trưng THÔ vào."
    if EXPORT_WITH_SCALER
    else "# Phải chuẩn hóa trước bằng standard_scaler.joblib:\n"
         "#     features = scaler.transform(features)"
)

print("\n=========================================================")
print("CÁCH DÙNG FILE ONNX")
print("=========================================================")
print(f"""
import json
import numpy as np
import onnxruntime as ort

with open("{os.path.basename(onnx_info_file)}", encoding="utf-8") as f:
    info = json.load(f)

session = ort.InferenceSession(
    "{os.path.basename(onnx_model_file)}",
    providers=["CPUExecutionProvider"]
)

{preprocessing_note}
# Thứ tự cột phải đúng như info["feature_names"], dtype float32.
features = np.zeros((1, {number_of_features}), dtype=np.float32)

probability = session.run(
    None,
    {{info["input_name"]: features}}
)[0].ravel()

is_ddos = probability >= info["threshold_default"]
print(probability, is_ddos)
""")

In [ ]:
%%time

# =========================================================
# BƯỚC 16: XUẤT MODEL SANG ĐỊNH DẠNG TFLITE
# =========================================================
# TFLite nhắm tới thiết bị biên: Raspberry Pi, router ARM, điện thoại.
# Runtime rất nhẹ (vài trăm KB) so với TensorFlow đầy đủ (hàng trăm MB).
#
# Cell xuất ba biến thể để so sánh kích thước / sai số / tốc độ:
#
#   float32       - Bản gốc, không mất mát. Dùng khi còn dư tài nguyên.
#   float16       - Trọng số nửa độ chính xác, nhỏ hơn ~2 lần.
#   dynamic_int8  - Lượng tử hóa động: trọng số int8, tính toán float.
#                   Nhỏ hơn ~3 lần, không cần dữ liệu hiệu chuẩn.
#
# CẠM BẪY LỚN NHẤT của TFLite: model xuất ra có batch cố định bằng 1.
# Muốn chạy theo lô phải gọi resize_tensor_input rồi allocate_tensors
# TRƯỚC khi nạp dữ liệu. Phần kiểm tra bên dưới làm đúng thứ tự này.

import json
import os
import shutil
import time

import flatbuffers
import numpy as np
import tensorflow as tf
from tensorflow.lite.python import schema_py_generated as tflite_schema

# Nên để trùng với EXPORT_WITH_SCALER ở Bước 15 để hai file hành xử giống nhau.
TFLITE_WITH_SCALER = True

# Các biến thể cần xuất.
TFLITE_VARIANTS = ["float32", "float16", "dynamic_int8"]

# Số mẫu test dùng để đối chiếu với Keras.
TFLITE_VERIFY_SAMPLE_SIZE = 5_000

# Sai số tối đa chấp nhận được cho bản float32 (bản lượng tử hóa được
# phép sai nhiều hơn, chỉ cần nhãn dự đoán không lệch quá nhiều).
TFLITE_TOLERANCE = 1e-4

# Tỷ lệ mẫu lệch nhãn tối đa chấp nhận được với bản lượng tử hóa.
TFLITE_MAX_LABEL_MISMATCH_RATE = 0.001

# Cấu hình đo tốc độ suy luận.
TFLITE_BENCHMARK_BATCH = 1_000
TFLITE_BENCHMARK_REPEATS = 5

try:
    TFLiteInterpreter = tf.lite.Interpreter
except AttributeError:
    from ai_edge_litert.interpreter import Interpreter as TFLiteInterpreter

# Muốn đổi tên tensor trong file .tflite thì phải mở flatbuffer ra sửa,
# nên cần schema đã sinh sẵn của TFLite. Cả hai gói dưới đây đều kèm nó.
import flatbuffers

try:
    from tensorflow.lite.python import schema_py_generated as tflite_schema
except ImportError:
    try:
        from ai_edge_litert import schema_py_generated as tflite_schema
    except ImportError:
        tflite_schema = None

# Thư mục chứa kết quả huấn luyện (nơi đọc model và scaler ra).
model_output_dir = os.path.abspath(
    os.path.join("output_after_preprocess", "dnn_model", SPLIT_MODE)
)

# Thư mục riêng cho các file đem đi triển khai, tách khỏi artefact huấn luyện.
tflite_output_dir = os.path.join(model_output_dir, "export", "tflite")
os.makedirs(tflite_output_dir, exist_ok=True)

tflite_info_file = os.path.join(
    tflite_output_dir,
    "tflite_deployment_info.json"
)
best_model_file = os.path.join(model_output_dir, "best_dnn_model.keras")

if "model" in globals():
    base_model = model
elif os.path.isfile(best_model_file):
    print(f"Đang tải model tốt nhất: {best_model_file}")
    base_model = tf.keras.models.load_model(best_model_file)
else:
    raise FileNotFoundError(
        "Không tìm thấy model để xuất. Hãy chạy lại Bước 11."
    )

if "scaler" not in globals():
    import joblib

    scaler_file = os.path.join(model_output_dir, "standard_scaler.joblib")
    if not os.path.isfile(scaler_file):
        raise FileNotFoundError(
            f"Không tìm thấy scaler: {scaler_file}. Hãy chạy lại Bước 8."
        )
    scaler = joblib.load(scaler_file)
    print(f"Đã nạp lại scaler từ: {scaler_file}")

scaler_mean = np.asarray(scaler.mean_, dtype=np.float32)
scaler_scale = np.asarray(scaler.scale_, dtype=np.float32)
number_of_features = len(feature_names)

if len(scaler_mean) != number_of_features:
    raise ValueError(
        f"Scaler có {len(scaler_mean)} feature nhưng feature_names có "
        f"{number_of_features}. Hai thứ phải khớp nhau."
    )

# ---------------------------------------------------------
# 1. Dựng model đem xuất
# ---------------------------------------------------------

if TFLITE_WITH_SCALER:
    # Rescaling giữ hệ số dưới dạng hằng số nên chúng được nhúng thẳng
    # vào file .tflite, không trở thành input thừa.
    raw_inputs = tf.keras.Input(
        shape=(number_of_features,),
        name="raw_features"
    )
    normalized = tf.keras.layers.Rescaling(
        scale=(1.0 / scaler_scale),
        offset=(-scaler_mean / scaler_scale),
        name="standard_scaler"
    )(raw_inputs)

    tflite_source_model = tf.keras.Model(
        inputs=raw_inputs,
        outputs=base_model(normalized),
        name="ddos_detector"
    )
    print("Chế độ xuất: TFLite nhận ĐẶC TRƯNG THÔ (đã nhúng sẵn scaler).")
else:
    tflite_source_model = base_model
    print("Chế độ xuất: TFLite nhận đặc trưng ĐÃ CHUẨN HÓA.")

print(f"Số đặc trưng đầu vào: {number_of_features}")

# Model phải được gọi ít nhất một lần trước khi chuyển đổi.
tflite_source_model.predict(
    np.zeros((1, number_of_features), dtype=np.float32),
    verbose=0
)

# ---------------------------------------------------------
# 1b. Đóng gói SavedModel với tên input/output do mình đặt
# ---------------------------------------------------------
# Nếu convert thẳng bằng from_keras_model(), converter tự sinh signature:
# khóa input lấy theo tên lớp tf.keras.Input ("raw_features"), khóa output
# bị đặt máy móc là "output_0". Muốn tên đúng ý thì phải tự khai báo
# signature: xuất SavedModel với endpoint nhận tham số tên "input", trả về
# dict khóa "output", rồi convert từ SavedModel đó.
#
# Sau khi convert, FlatBuffer sẽ được hậu xử lý để cả khóa signature lẫn
# trường name của tensor mà Netron hiển thị đều đúng là "input" / "output".

TFLITE_INPUT_NAME = "input"
TFLITE_OUTPUT_NAME = "output"
TFLITE_SIGNATURE_KEY = "serving_default"

tflite_saved_model_dir = os.path.join(tflite_output_dir, "_saved_model_tmp")
if os.path.isdir(tflite_saved_model_dir):
    shutil.rmtree(tflite_saved_model_dir)

serving_input_signature = [
    tf.TensorSpec(
        [None, number_of_features],
        tf.float32,
        name=TFLITE_INPUT_NAME
    )
]

if hasattr(getattr(tf.keras, "export", None), "ExportArchive"):
    # Keras 3: biến của model nằm trong KerasVariable nên tf.Module không
    # tự theo dõi được. ExportArchive.track() làm đúng việc đó.
    def serving_endpoint(input):
        return {
            TFLITE_OUTPUT_NAME: tflite_source_model(input, training=False)
        }

    export_archive = tf.keras.export.ExportArchive()
    export_archive.track(tflite_source_model)
    export_archive.add_endpoint(
        name=TFLITE_SIGNATURE_KEY,
        fn=serving_endpoint,
        input_signature=serving_input_signature
    )
    export_archive.write_out(tflite_saved_model_dir)
else:
    # Keras 2: model đã là tf.Module nên lưu thẳng được.
    class ServingModule(tf.Module):
        """Bọc model để tự quyết định tên input/output của signature."""

        def __init__(self, keras_model):
            super().__init__()
            self.keras_model = keras_model

        @tf.function(input_signature=serving_input_signature)
        def serving_endpoint(self, input):
            return {
                TFLITE_OUTPUT_NAME: self.keras_model(input, training=False)
            }

    serving_module = ServingModule(tflite_source_model)
    tf.saved_model.save(
        serving_module,
        tflite_saved_model_dir,
        signatures={TFLITE_SIGNATURE_KEY: serving_module.serving_endpoint}
    )

print(f"SavedModel trung gian: {tflite_saved_model_dir}")
print(
    f"Tên signature sẽ dùng: input='{TFLITE_INPUT_NAME}', "
    f"output='{TFLITE_OUTPUT_NAME}'"
)

# ---------------------------------------------------------
# 2. Chuẩn bị dữ liệu đối chiếu
# ---------------------------------------------------------

verify_size = min(TFLITE_VERIFY_SAMPLE_SIZE, len(X_test_dnn))
verify_rng = np.random.default_rng(42)
verify_index = verify_rng.choice(len(X_test_dnn), size=verify_size, replace=False)

scaled_features = np.ascontiguousarray(
    X_test_dnn[verify_index],
    dtype=np.float32
)

if TFLITE_WITH_SCALER:
    # Tái tạo đặc trưng thô rồi chuẩn hóa lại bằng đúng công thức, để hai
    # phía cùng xuất phát từ một điểm.
    raw_features = scaled_features * scaler_scale + scaler_mean
    tflite_input_values = raw_features
    keras_input_values = (raw_features - scaler_mean) / scaler_scale
else:
    tflite_input_values = scaled_features
    keras_input_values = scaled_features

tflite_input_values = np.ascontiguousarray(
    tflite_input_values,
    dtype=np.float32
)

keras_probability = base_model.predict(
    keras_input_values,
    batch_size=4096,
    verbose=0
).ravel()

thresholds_to_check = {"0.5": 0.5}
if "best_threshold" in globals():
    thresholds_to_check["tối ưu F1"] = float(best_threshold)


# ---------------------------------------------------------
# 3. Hàm chuyển đổi và kiểm tra
# ---------------------------------------------------------

def rename_tflite_io(tflite_bytes):
    """Đổi tên tensor input/output ngay trong flatbuffer .tflite.

    Converter luôn tự đặt tên tensor là "serving_default_<khóa>:0" và
    "StatefulPartitionedCall:0", không có tùy chọn nào tắt được. Muốn
    trường name đúng bằng "input"/"output" thì chỉ còn cách mở flatbuffer
    ra sửa rồi đóng gói lại.

    An toàn: mọi thứ khác trong file (node, signature, buffer) tham chiếu
    tensor theo CHỈ SỐ chứ không theo tên, nên đổi tên không làm hỏng gì.
    """
    if tflite_schema is None:
        raise RuntimeError(
            "Không nạp được schema flatbuffer của TFLite "
            "(tensorflow.lite.python.schema_py_generated hoặc "
            "ai_edge_litert.schema_py_generated) nên không đổi được tên "
            "tensor. Cài: pip install tensorflow hoặc ai-edge-litert."
        )

    model = tflite_schema.ModelT.InitFromObj(
        tflite_schema.Model.GetRootAsModel(bytearray(tflite_bytes), 0)
    )

    for subgraph in model.subgraphs:
        for position, tensor_index in enumerate(subgraph.inputs):
            new_name = (
                TFLITE_INPUT_NAME
                if len(subgraph.inputs) == 1
                else f"{TFLITE_INPUT_NAME}_{position}"
            )
            subgraph.tensors[tensor_index].name = new_name.encode()

        for position, tensor_index in enumerate(subgraph.outputs):
            new_name = (
                TFLITE_OUTPUT_NAME
                if len(subgraph.outputs) == 1
                else f"{TFLITE_OUTPUT_NAME}_{position}"
            )
            subgraph.tensors[tensor_index].name = new_name.encode()

    builder = flatbuffers.Builder(1024)
    # "TFL3" là file_identifier bắt buộc, thiếu nó interpreter sẽ không nạp.
    builder.Finish(model.Pack(builder), b"TFL3")

    return bytes(builder.Output())


def build_converter(variant):
    """Tạo converter đã cấu hình theo biến thể lượng tử hóa."""
    converter = tf.lite.TFLiteConverter.from_saved_model(
        tflite_saved_model_dir,
        signature_keys=[TFLITE_SIGNATURE_KEY]
    )

    if variant == "float16":
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
    elif variant == "dynamic_int8":
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    elif variant != "float32":
        raise ValueError(f"Biến thể không hợp lệ: {variant}")

    return converter


def rename_tflite_io_tensors(model_bytes, input_name, output_name):
    """Đổi trường name của tensor I/O ngay trong FlatBuffer TFLite."""
    model_object = tflite_schema.Model.GetRootAsModel(model_bytes, 0)
    model = tflite_schema.ModelT.InitFromObj(model_object)

    if not model.subgraphs:
        raise RuntimeError("File TFLite không có subgraph.")

    main_graph = model.subgraphs[0]
    if len(main_graph.inputs) != 1 or len(main_graph.outputs) != 1:
        raise RuntimeError(
            "Chỉ hỗ trợ model TFLite có đúng 1 input và 1 output."
        )

    main_graph.tensors[main_graph.inputs[0]].name = input_name.encode("utf-8")
    main_graph.tensors[main_graph.outputs[0]].name = output_name.encode("utf-8")

    builder = flatbuffers.Builder(0)
    model_offset = model.Pack(builder)
    builder.Finish(model_offset, file_identifier=b"TFL3")
    return bytes(builder.Output())


def run_tflite(model_path, features):
    """Chạy suy luận TFLite theo lô, trả về xác suất."""
    interpreter = TFLiteInterpreter(model_path=model_path)

    input_detail = interpreter.get_input_details()[0]
    default_shape = list(input_detail["shape"])

    # Batch mặc định của TFLite là 1 nên phải đổi kích thước trước.
    interpreter.resize_tensor_input(
        input_detail["index"],
        [len(features), number_of_features]
    )
    interpreter.allocate_tensors()

    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]

    interpreter.set_tensor(input_detail["index"], features)
    interpreter.invoke()

    return (
        interpreter.get_tensor(output_detail["index"]).ravel(),
        default_shape,
        input_detail["name"],
        output_detail["name"]
    )


def benchmark_tflite(model_path, features):
    """Đo tốc độ suy luận, trả về số flow xử lý mỗi giây."""
    interpreter = TFLiteInterpreter(model_path=model_path)
    input_detail = interpreter.get_input_details()[0]

    interpreter.resize_tensor_input(
        input_detail["index"],
        [len(features), number_of_features]
    )
    interpreter.allocate_tensors()

    input_detail = interpreter.get_input_details()[0]

    # Chạy nóng máy trước khi đo
    interpreter.set_tensor(input_detail["index"], features)
    interpreter.invoke()

    start_time = time.perf_counter()
    for _ in range(TFLITE_BENCHMARK_REPEATS):
        interpreter.set_tensor(input_detail["index"], features)
        interpreter.invoke()
    elapsed = time.perf_counter() - start_time

    total_samples = TFLITE_BENCHMARK_REPEATS * len(features)
    return total_samples / elapsed if elapsed > 0 else float("nan")


# ---------------------------------------------------------
# 4. Xuất từng biến thể
# ---------------------------------------------------------

benchmark_features = np.ascontiguousarray(
    tflite_input_values[:min(TFLITE_BENCHMARK_BATCH, verify_size)],
    dtype=np.float32
)

variant_rows = []
variant_details = {}

for variant in TFLITE_VARIANTS:
    suffix = "" if variant == "float32" else f"_{variant}"
    tflite_file = os.path.join(
        tflite_output_dir,
        f"ddos_dnn_model{suffix}.tflite"
    )

    print(f"\n--- Đang xuất biến thể: {variant} ---")

    tflite_bytes = rename_tflite_io(build_converter(variant).convert())
    tflite_bytes = rename_tflite_io_tensors(
        tflite_bytes,
        TFLITE_INPUT_NAME,
        TFLITE_OUTPUT_NAME
    )

    with open(tflite_file, "wb") as tflite_handle:
        tflite_handle.write(tflite_bytes)

    size_kb = os.path.getsize(tflite_file) / 1024

    tflite_probability, default_shape, input_name, output_name = run_tflite(
        tflite_file,
        tflite_input_values
    )

    if input_name != TFLITE_INPUT_NAME or output_name != TFLITE_OUTPUT_NAME:
        raise RuntimeError(
            "Tên tensor TFLite không đúng sau khi xuất: "
            f"input='{input_name}', output='{output_name}'."
        )

    if input_name != TFLITE_INPUT_NAME or output_name != TFLITE_OUTPUT_NAME:
        raise RuntimeError(
            f"Tên tensor trong file là '{input_name}' / '{output_name}', "
            f"đáng lẽ phải là '{TFLITE_INPUT_NAME}' / '{TFLITE_OUTPUT_NAME}'."
        )

    max_difference = float(np.abs(tflite_probability - keras_probability).max())
    mean_difference = float(np.abs(tflite_probability - keras_probability).mean())

    mismatch_by_threshold = {}
    worst_mismatch_rate = 0.0

    for threshold_name, threshold_value in thresholds_to_check.items():
        keras_labels = (keras_probability >= threshold_value).astype(np.int8)
        tflite_labels = (tflite_probability >= threshold_value).astype(np.int8)
        mismatch = int((keras_labels != tflite_labels).sum())

        mismatch_by_threshold[threshold_name] = mismatch
        worst_mismatch_rate = max(worst_mismatch_rate, mismatch / verify_size)

    flows_per_second = benchmark_tflite(tflite_file, benchmark_features)

    print(f"  Kích thước      : {size_kb:.1f} KB")
    print(f"  Tên input/output: {input_name} / {output_name}")
    print(f"  Input mặc định  : {default_shape} (batch cố định = 1)")
    print(f"  Sai lệch lớn nhất: {max_difference:.3e}")
    for threshold_name, mismatch in mismatch_by_threshold.items():
        print(
            f"  Lệch nhãn @ {threshold_name}: {mismatch:,} / {verify_size:,}"
        )
    print(f"  Tốc độ          : {flows_per_second:,.0f} flow/giây")

    if variant == "float32" and max_difference > TFLITE_TOLERANCE:
        raise RuntimeError(
            f"Bản float32 sai lệch {max_difference:.3e}, vượt ngưỡng "
            f"{TFLITE_TOLERANCE:.0e}. KHÔNG dùng file này."
        )

    if worst_mismatch_rate > TFLITE_MAX_LABEL_MISMATCH_RATE:
        print(
            f"  CẢNH BÁO: tỷ lệ lệch nhãn {worst_mismatch_rate:.4%} vượt "
            f"mức chấp nhận {TFLITE_MAX_LABEL_MISMATCH_RATE:.2%}."
        )
        print(
            "    Không dùng biến thể này ở dạng hiện tại. Cách khắc phục: "
            "đặt TFLITE_WITH_SCALER = False rồi xuất lại, và tự áp scaler "
            "ở phía ứng dụng."
        )
        if TFLITE_WITH_SCALER:
            print(
                "    Lý do thường gặp: hằng số mean/scale của lớp Rescaling "
                "trải trên nhiều bậc độ lớn nên bị mất chính xác khi lượng "
                "tử hóa (float16 chỉ biểu diễn được tới 65.504)."
            )

    variant_rows.append({
        "Biến thể": variant,
        "File": os.path.basename(tflite_file),
        "Kích thước (KB)": round(size_kb, 1),
        "Sai lệch lớn nhất": max_difference,
        "Sai lệch trung bình": mean_difference,
        "Lệch nhãn @0.5": mismatch_by_threshold["0.5"],
        "Tốc độ (flow/giây)": round(flows_per_second),
    })

    variant_details[variant] = {
        "file": os.path.basename(tflite_file),
        "size_kb": round(size_kb, 1),
        "max_absolute_difference": max_difference,
        "mean_absolute_difference": mean_difference,
        "label_mismatch": mismatch_by_threshold,
        "flows_per_second": round(flows_per_second),
        "input_name": input_name,
        "output_name": output_name,
    }

# ---------------------------------------------------------
# 5. Bảng so sánh
# ---------------------------------------------------------

import pandas as pd

comparison_df = pd.DataFrame(variant_rows)

print("\n=========================================================")
print("SO SÁNH CÁC BIẾN THỂ TFLITE")
print("=========================================================")
display(comparison_df)

print(
    "\nLưu ý: tốc độ đo trên máy đang chạy notebook, chỉ mang tính tham "
    "khảo để so sánh tương đối giữa các biến thể. Trên thiết bị biên "
    "(ARM) con số sẽ thấp hơn nhiều."
)

# ---------------------------------------------------------
# 6. Lưu thông tin triển khai
# ---------------------------------------------------------

# Đọc lại signature từ chính file đã xuất để chắc chắn tên đúng.
tflite_signature = TFLiteInterpreter(
    model_path=os.path.join(tflite_output_dir, "ddos_dnn_model.tflite")
).get_signature_list()

print(f"\nSignature trong file .tflite: {tflite_signature}")

deployment_info = {
    "format": "tflite",
    "includes_scaler": bool(TFLITE_WITH_SCALER),
    "input_description": (
        "Đặc trưng THÔ theo đúng thứ tự feature_names"
        if TFLITE_WITH_SCALER
        else "Đặc trưng ĐÃ chuẩn hóa bằng standard_scaler.joblib"
    ),
    "input_dtype": "float32",
    "signature_key": TFLITE_SIGNATURE_KEY,
    "signature_input_name": TFLITE_INPUT_NAME,
    "signature_output_name": TFLITE_OUTPUT_NAME,
    "default_batch_size": 1,
    "batch_note": (
        "Phải gọi resize_tensor_input rồi allocate_tensors trước khi nạp "
        "dữ liệu nếu muốn chạy theo lô."
    ),
    "num_features": number_of_features,
    "feature_names": feature_names,
    "label_mapping": {name: int(code) for name, code in label_mapping.items()},
    "split_mode": SPLIT_MODE,
    "split_description": SPLIT_DESCRIPTION,
    "threshold_default": 0.5,
    "threshold_tuned": (
        float(best_threshold) if "best_threshold" in globals() else None
    ),
    "verification_sample_size": int(verify_size),
    "variants": variant_details,
    "versions": {
        "tensorflow": tf.__version__,
        "keras": tf.keras.__version__,
    },
}

with open(tflite_info_file, "w", encoding="utf-8") as info_handle:
    json.dump(deployment_info, info_handle, ensure_ascii=False, indent=2)

comparison_df.to_csv(
    os.path.join(tflite_output_dir, "tflite_variants_comparison.csv"),
    index=False,
    encoding="utf-8"
)

# SavedModel trung gian chỉ phục vụ việc convert, không đem đi triển khai.
shutil.rmtree(tflite_saved_model_dir, ignore_errors=True)

print(f"\nThư mục xuất TFLite: {tflite_output_dir}")
print(f"Đã lưu thông tin triển khai: {tflite_info_file}")

# ---------------------------------------------------------
# 7. Mẫu code sử dụng
# ---------------------------------------------------------

preprocessing_note = (
    "# File TFLite đã nhúng sẵn scaler: đưa thẳng đặc trưng THÔ vào."
    if TFLITE_WITH_SCALER
    else "# Phải chuẩn hóa trước bằng standard_scaler.joblib."
)

print("\n=========================================================")
print("CÁCH DÙNG FILE TFLITE")
print("=========================================================")
print(f"""
import json
import numpy as np

# Trên thiết bị biên chỉ cần gói runtime nhẹ, không cần TensorFlow:
#     pip install ai-edge-litert
# from ai_edge_litert.interpreter import Interpreter
import tensorflow as tf
Interpreter = tf.lite.Interpreter

with open("tflite_deployment_info.json", encoding="utf-8") as f:
    info = json.load(f)

interpreter = Interpreter(
    model_path="ddos_dnn_model.tflite",
    num_threads=4
)

{preprocessing_note}
# Thứ tự cột phải đúng như info["feature_names"], dtype float32.
features = np.zeros(({TFLITE_BENCHMARK_BATCH}, {number_of_features}), dtype=np.float32)

# Model có signature "serving_default" với input="input", output="output".
# Cách gọn nhất, gọi thẳng theo tên:
#     runner = interpreter.get_signature_runner()
#     probability = runner(input=features)["output"].ravel()
# Cách theo chỉ số bên dưới không phụ thuộc tên, dùng được ở mọi phiên bản.

# BẮT BUỘC: đổi kích thước batch rồi mới cấp phát tensor.
input_index = interpreter.get_input_details()[0]["index"]
interpreter.resize_tensor_input(input_index, list(features.shape))
interpreter.allocate_tensors()

input_index = interpreter.get_input_details()[0]["index"]
output_index = interpreter.get_output_details()[0]["index"]

interpreter.set_tensor(input_index, features)
interpreter.invoke()

probability = interpreter.get_tensor(output_index).ravel()
is_ddos = probability >= info["threshold_default"]
""")